# CineTrope — Pipeline di costruzione dell'ontologia OWL 2 DL

Questo notebook **documenta e riproduce interamente in Python (`rdflib`)** la costruzione dell'ontologia OWL 2 DL ($\mathcal{SROIQ}(\mathcal{D})$) di **CineTrope**, il reasoning con **HermiT via OWLReady2** e la verifica tramite query **SPARQL**.

## Architettura della pipeline

```
data/raw/films.json           → selezione top-100 film per popolarità TMDB
    ↓
TMDB (locale)                 → TBox completa (OWL 2 DL) + ABox con rdflib
Wikidata SPARQL (+ cache)     → premi Oscar Best Picture + link owl:sameAs (wd:)
DBpedia SPARQL (+ cache)      → date/anni di nascita registi + link owl:sameAs (dbr:)
    ↓
data/ontology/core.ttl        ← Turtle completo (TBox + ABox + 6 regole SWRL)
data/ontology/core.owl        ← RDF/XML nativo Protégé
data/ontology/core.swrl.n3    ← Regole SWRL isolate in N3
    ↓
OWLReady2 + HermiT            → reasoning OWL 2 DL (consistenza, classificazione, property chain, SWRL)
    ↓
data/generated/inferred.ttl   ← triple inferite da HermiT materializzate
data/generated/store.trig     ← dataset a named graph (TriG) per FastAPI
```

**Dipendenze**: `rdflib`, `owlready2`, `SPARQLWrapper`, `tqdm`  
**Java**: HermiT richiede Java ≥ 11 nel PATH oppure una JRE locale in `.venv/jre`.  
**Riferimento accademico**: vedi `notebooks/CONFRONTO_TEORIA.md` per il confronto completo con le Lezioni 1–12 e gli esercizi pratici in `teoria/`.

In [1]:
# ── Cella 0: Dipendenze, path e namespace ─────────────────────────────────
import json
import os
import pathlib
import shutil
import sys
import tempfile
import time
import unicodedata
from collections import Counter, defaultdict
from decimal import Decimal
from urllib.parse import quote

from rdflib import (
    BNode, Dataset, Graph, Literal, Namespace,
    OWL, RDF, RDFS, URIRef, XSD,
)
from rdflib.collection import Collection
from tqdm.auto import tqdm

# Root del progetto (funziona sia dalla root sia da notebooks/)
_cwd = pathlib.Path.cwd().resolve()
if (_cwd / "data" / "raw" / "films.json").exists():
    ROOT = _cwd
elif (_cwd.parent / "data" / "raw" / "films.json").exists():
    ROOT = _cwd.parent
else:
    ROOT = pathlib.Path("__file__").resolve().parent.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW_DIR       = ROOT / "data" / "raw"
ONTOLOGY_DIR  = ROOT / "data" / "ontology"
GENERATED_DIR = ROOT / "data" / "generated"
GENERATED_DIR.mkdir(parents=True, exist_ok=True)
ONTOLOGY_DIR.mkdir(parents=True, exist_ok=True)

CORE_TTL      = ONTOLOGY_DIR / "core.ttl"
CORE_OWL      = ONTOLOGY_DIR / "core.owl"
CORE_SWRL_N3  = ONTOLOGY_DIR / "core.swrl.n3"
INFERRED_TTL  = GENERATED_DIR / "inferred.ttl"
STORE_PATH    = GENERATED_DIR / "store.trig"

# Namespace principali e vocabolari esterni (L3, L5-6, L11)
CT      = Namespace("http://example.org/cinetrope#")
SCHEMA  = Namespace("http://schema.org/")
DBO     = Namespace("http://dbpedia.org/ontology/")
DBR     = Namespace("http://dbpedia.org/resource/")
WDT     = Namespace("http://www.wikidata.org/prop/direct/")
WD      = Namespace("http://www.wikidata.org/entity/")
FOAF    = Namespace("http://xmlns.com/foaf/0.1/")
SKOS    = Namespace("http://www.w3.org/2004/02/skos/core#")
SWRL    = Namespace("http://www.w3.org/2003/11/swrl#")
SWRLA   = Namespace("http://swrl.stanford.edu/ontologies/3.3/swrla.owl#")
DCTERMS = Namespace("http://purl.org/dc/terms/")

print("✓ Import completati")
print(f"  ROOT = {ROOT}")

✓ Import completati
  ROOT = C:\uni\CineTrope


## Cella 1 — Selezione corpus: top-100 film per popolarità TMDB

Da `data/raw/films.json` (500 film scaricati da TMDB) selezioniamo i **100 film con popolarità TMDB più alta**. La dimensione ristretta (100 film, ~1.000 persone, 17 generi, ~880 tropi) rende il grafo ispezionabile agevolmente in **Protégé** e permette al reasoner hypertableau **HermiT** di completare il reasoning $\mathcal{SROIQ}(\mathcal{D})$ + SWRL in pochi secondi.

In [2]:
# ── Cella 1: Selezione corpus ──────────────────────────────────────────────
films_all = json.loads((RAW_DIR / "films.json").read_text(encoding="utf-8"))
films_all = [f for f in films_all if f.get("type") == "Film"]
top100 = sorted(films_all, key=lambda f: f.get("popularity", 0), reverse=True)[:100]

print(f"Film totali nel dataset: {len(films_all)}")
print(f"Film selezionati (top-100 per popolarità): {len(top100)}")
print("\nTop 10:")
for i, f in enumerate(top100[:10], 1):
    print(f"  {i:2d}. {f['title']} ({f.get('year','?')}) — popolarità {f.get('popularity',0):.1f}")

genres_corpus = Counter(g for f in top100 for g in f.get("genres", []))
kws_corpus    = Counter(k["id"] for f in top100 for k in f.get("keywords", []))
print(f"\nGeneri distinti nel corpus: {len(genres_corpus)}")
print(f"Keyword/tropi distinti:     {len(kws_corpus)}")
print(f"Registi distinti:           {len({d for f in top100 for d in f.get('directors',[])})}")
print(f"Attori distinti (top-10):   {len({a for f in top100 for a in f.get('cast',[])[:10]})}")

Film totali nel dataset: 500
Film selezionati (top-100 per popolarità): 100

Top 10:
   1. Heart of the Beast - Nel profondo selvaggio (2026) — popolarità 507.5
   2. Odissea (2026) — popolarità 478.6
   3. Mammina (2026) — popolarità 411.8
   4. Resident Evil (2026) — popolarità 401.0
   5. Spider-Man: Brand New Day (2026) — popolarità 328.3
   6. Hope (2026) — popolarità 250.3
   7. L'ultima missione: Project Hail Mary (2026) — popolarità 240.6
   8. Never Surrender (2026) — popolarità 234.2
   9. A Story of Derry (2026) — popolarità 223.3
  10. Apocalipsis Z: Il nuovo mondo (2026) — popolarità 187.0

Generi distinti nel corpus: 18
Keyword/tropi distinti:     879
Registi distinti:           117
Attori distinti (top-10):   909


## Cella 2 — Costruzione TBox OWL 2 DL con `rdflib`

Costruiamo la **TBox** interamente via codice Python senza scrivere Turtle a mano. Ogni funzione helper modella un costruttore formale di **OWL 2 DL ($\mathcal{SROIQ}(\mathcal{D})$)**:

- **Classi primitive, gerarchie e disgiunzioni**: `rdfs:subClassOf`, `owl:disjointWith`, `owl:AllDisjointClasses`, `owl:disjointUnionOf` (*covering + disjointness* su `OperaAudiovisiva ≡ Film ⊔ SerieTV`).
- **Condizioni necessarie universali**: `owl:allValuesFrom` su `Film`.
- **Annotazioni (pattern L10 `ActorsFilms` compatibile OWL 2 DL)**: `ct:displayTitle a owl:AnnotationProperty ; rdfs:subPropertyOf rdfs:label` (in OWL 2 DL le `DatatypeProperty` sono disgiunte dalle `AnnotationProperty`, a differenza di RDFS puro).
- **Object properties**: `rdfs:subPropertyOf` (pattern `Library` L11–12), `owl:inverseOf`, `owl:SymmetricProperty`, `owl:TransitiveProperty`, `owl:AsymmetricProperty`, `owl:IrreflexiveProperty` (separando ruoli semplici `directSequelOf` da ruoli non semplici `sequelOf`), `owl:propertyChainAxiom`.
- **Data properties**: `owl:FunctionalProperty` + pattern **L10 `NaviFinal`** anti-contaminazione di dominio (`countryIsoCode` e `languageIsoCode` sottoproprietà della superproprietà neutra `isoCode`).
- **16 classi definite (`owl:equivalentClass`)** che coprono i costruttori OWL 2 DL *nominal-free* (mantenendo HermiT 4× più veloce rispetto al tableau con merging dei nominali): `datatype restriction` (`minInclusive`, `maxInclusive`, `minExclusive`, `maxExclusive`), `someValuesFrom`, `minQualifiedCardinality` (+ `AllDifferent`), `hasValue`, `intersectionOf`, `unionOf`, `complementOf` (+ `FunctionalProperty` come dimostrazione **OWA**).

In [3]:
# ── Cella 2: TBox — Metadati, helper OWL 2 DL e classi primitive ──────────
g = Graph()
for prefix, ns in [
    ("ct", CT), ("owl", OWL), ("rdf", RDF), ("rdfs", RDFS), ("xsd", XSD),
    ("schema", SCHEMA), ("dbo", DBO), ("dbr", DBR), ("wdt", WDT), ("wd", WD),
    ("foaf", FOAF), ("skos", SKOS), ("swrl", SWRL), ("swrla", SWRLA),
    ("dcterms", DCTERMS),
]:
    g.bind(prefix, ns, override=True, replace=True)

# ─── Metadati ontologia (Dublin Core + OWL, stile OntoComedySources L11) ──
onto_uri = URIRef("http://example.org/cinetrope")
g.add((onto_uri, RDF.type, OWL.Ontology))
g.add((onto_uri, OWL.versionIRI, URIRef("http://example.org/cinetrope/3.1.0")))
g.add((onto_uri, OWL.versionInfo, Literal("3.1.0")))
g.add((onto_uri, RDFS.label, Literal("CineTrope - Ontologia OWL 2 DL", lang="it")))
g.add((onto_uri, RDFS.label, Literal("CineTrope - OWL 2 DL Ontology", lang="en")))
g.add((onto_uri, RDFS.comment, Literal(
    "Ontologia OWL 2 DL (SROIQ(D)) per raccomandazioni cinematografiche spiegabili. "
    "Corpus: top-100 film TMDB arricchiti da Wikidata e DBpedia. Reasoning: HermiT + SWRL DL-safe.",
    lang="it",
)))
g.add((onto_uri, DCTERMS.title, Literal("CineTrope Knowledge Graph Ontology", lang="en")))
g.add((onto_uri, DCTERMS.creator, Literal("Progetto Universitario Semantic Web — Università di Catania", lang="it")))
g.add((onto_uri, DCTERMS.description, Literal(
    "Ontologia prodotta programmaticamente con rdflib: copre RDFS, OWL 2 DL (SROIQ(D)), "
    "Linked Open Data (owl:sameAs verso DBpedia/Wikidata) e regole SWRL DL-safe.",
    lang="it",
)))
g.add((onto_uri, DCTERMS.created, Literal("2026-11-21", datatype=XSD.string)))

# Dichiarazione AnnotationProperty esterne/locali per conformità OWL 2 DL in Protégé
for ann_prop in (DCTERMS.title, DCTERMS.creator, DCTERMS.description, DCTERMS.created,
                 SKOS.prefLabel, SKOS.altLabel, SWRLA.isRuleEnabled):
    g.add((ann_prop, RDF.type, OWL.AnnotationProperty))

# Pattern L10 ActorsFilms in OWL 2 DL: subproperty di rdfs:label come AnnotationProperty
g.add((CT.displayTitle, RDF.type, OWL.AnnotationProperty))
g.add((CT.displayTitle, RDFS.subPropertyOf, RDFS.label))
g.add((CT.displayTitle, RDFS.label, Literal("titolo visualizzato", lang="it")))
g.add((CT.displayTitle, RDFS.label, Literal("display title", lang="en")))
g.add((CT.displayTitle, RDFS.comment, Literal(
    "Sottoproprietà di rdfs:label (pattern Lezione 10 ActorsFilms) tipizzata come "
    "owl:AnnotationProperty per rispettare la separazione OWL 2 DL tra annotazioni e data property.",
    lang="it",
)))


# ─── Helper programmatici per costruttori OWL 2 DL ────────────────────────
def add_class(uri, label_it, label_en, comment_it=None, subclass_of=None, disjoint_with=None):
    """Aggiunge una classe primitiva al grafo."""
    g.add((uri, RDF.type, OWL.Class))
    g.add((uri, RDFS.label, Literal(label_it, lang="it")))
    g.add((uri, RDFS.label, Literal(label_en, lang="en")))
    if comment_it:
        g.add((uri, RDFS.comment, Literal(comment_it, lang="it")))
    if subclass_of:
        g.add((uri, RDFS.subClassOf, subclass_of))
    if disjoint_with:
        g.add((uri, OWL.disjointWith, disjoint_with))
    return uri


def owl_restriction(property_uri, restriction_type, value):
    """Crea un nodo anonimo owl:Restriction (someValuesFrom, allValuesFrom, hasValue)."""
    r = BNode()
    g.add((r, RDF.type, OWL.Restriction))
    g.add((r, OWL.onProperty, property_uri))
    g.add((r, restriction_type, value))
    return r


def owl_intersection(*class_nodes):
    """Crea owl:intersectionOf su una lista di classi/restrizioni."""
    anon = BNode()
    g.add((anon, RDF.type, OWL.Class))
    coll = BNode()
    Collection(g, coll, list(class_nodes))
    g.add((anon, OWL.intersectionOf, coll))
    return anon


def owl_union(*class_nodes):
    """Crea owl:unionOf su una lista di classi/restrizioni."""
    anon = BNode()
    g.add((anon, RDF.type, OWL.Class))
    coll = BNode()
    Collection(g, coll, list(class_nodes))
    g.add((anon, OWL.unionOf, coll))
    return anon


def owl_complement(class_node):
    """Crea owl:complementOf (negazione DL ¬C)."""
    anon = BNode()
    g.add((anon, RDF.type, OWL.Class))
    g.add((anon, OWL.complementOf, class_node))
    return anon


def owl_one_of(*individual_uris):
    """Crea classe enumerata owl:oneOf (nominale O in SROIQ)."""
    anon = BNode()
    g.add((anon, RDF.type, OWL.Class))
    coll = BNode()
    Collection(g, coll, list(individual_uris))
    g.add((anon, OWL.oneOf, coll))
    return anon


def owl_datatype_restriction(datatype, facet_or_list, value_literal=None):
    """Crea rdfs:Datatype con owl:withRestrictions (uno o più facet XSD)."""
    dt = BNode()
    g.add((dt, RDF.type, RDFS.Datatype))
    g.add((dt, OWL.onDatatype, datatype))
    pairs = facet_or_list if isinstance(facet_or_list, (list, tuple)) else [(facet_or_list, value_literal)]
    facet_nodes = []
    for facet, val in pairs:
        fn = BNode()
        g.add((fn, facet, val))
        facet_nodes.append(fn)
    coll = BNode()
    Collection(g, coll, facet_nodes)
    g.add((dt, OWL.withRestrictions, coll))
    return dt


def add_min_qualified_cardinality(property_uri, n, on_class):
    """Crea una restrizione owl:minQualifiedCardinality."""
    r = BNode()
    g.add((r, RDF.type, OWL.Restriction))
    g.add((r, OWL.onProperty, property_uri))
    g.add((r, OWL.minQualifiedCardinality, Literal(n, datatype=XSD.nonNegativeInteger)))
    g.add((r, OWL.onClass, on_class))
    return r


def add_defined_class(cls_uri, label_it, label_en, comment_it, subclass_of, equivalent_to):
    """Aggiunge una classe definita con owl:equivalentClass."""
    g.add((cls_uri, RDF.type, OWL.Class))
    g.add((cls_uri, RDFS.subClassOf, subclass_of))
    g.add((cls_uri, RDFS.label, Literal(label_it, lang="it")))
    g.add((cls_uri, RDFS.label, Literal(label_en, lang="en")))
    g.add((cls_uri, RDFS.comment, Literal(comment_it, lang="it")))
    g.add((cls_uri, OWL.equivalentClass, equivalent_to))
    return cls_uri


def add_disjoint_union(cls_uri, member_classes):
    """Aggiunge un assioma di covering + disgiunzione owl:disjointUnionOf."""
    coll = BNode()
    Collection(g, coll, list(member_classes))
    g.add((cls_uri, OWL.disjointUnionOf, coll))


def add_has_key(cls_uri, prop_uris):
    """Aggiunge una chiave OWL 2 (owl:hasKey) per identificazione univoca sotto No-UNA."""
    coll = BNode()
    Collection(g, coll, list(prop_uris))
    g.add((cls_uri, OWL.hasKey, coll))


# ─── Classi primitive ──────────────────────────────────────────────────
print("[TBox] Classi primitive, disjointUnionOf e AllDisjointClasses...")
add_class(CT.OperaAudiovisiva, "Opera audiovisiva", "Audiovisual work",
          "Superclasse comune di Film e SerieTV (partizionata da owl:disjointUnionOf).")
add_class(CT.Film, "Film", "Film", "Un film cinematografico.",
          subclass_of=CT.OperaAudiovisiva, disjoint_with=CT.SerieTV)
add_class(CT.SerieTV, "Serie TV", "TV series",
          "Una serie televisiva (disgiunta da Film).",
          subclass_of=CT.OperaAudiovisiva)
add_disjoint_union(CT.OperaAudiovisiva, [CT.Film, CT.SerieTV])

add_class(CT.Persona, "Persona", "Person", "Persona coinvolta nella produzione cinematografica.")
add_class(CT.Regista, "Regista", "Director",
          "Persona che ha diretto almeno un'opera.", subclass_of=CT.Persona)
add_class(CT.Attore, "Attore/Attrice", "Actor",
          "Persona che ha recitato in almeno un'opera.", subclass_of=CT.Persona)
add_class(CT.Genere, "Genere cinematografico", "Film genre",
          "Genere cinematografico modellato come individuo (con gerarchia transitiva subGenreOf).")
add_class(CT.Tropo, "Tropo narrativo", "Narrative trope",
          "Keyword TMDB usata come tropo narrativo condivisibile tra film.")
add_class(CT.Premio, "Premio cinematografico", "Film award",
          "Riconoscimento ufficiale assegnato a film o persone.")
add_class(CT.Paese, "Paese", "Country", "Paese di produzione cinematografica.")
add_class(CT.Lingua, "Lingua", "Language", "Lingua originale dell'opera audiovisiva.")
add_class(CT.Studio, "Studio di produzione", "Production company", "Casa o studio di produzione.")
add_class(CT.Collezione, "Collezione/franchise", "Collection/franchise", "Saga o franchise cinematografico.")
add_class(CT.Fonte, "Fonte", "Source",
          "Named graph contenente le asserzioni provenienti da una specifica sorgente dati.")

# Alias terminologico (equivalenza nominale tra classi, L12 Researcher/Analyst)
g.add((CT.Movie, RDF.type, OWL.Class))
g.add((CT.Movie, OWL.equivalentClass, CT.Film))
g.add((CT.Movie, RDFS.label, Literal("Film (alias)", lang="it")))
g.add((CT.Movie, RDFS.label, Literal("Movie", lang="en")))
g.add((CT.Movie, RDFS.comment, Literal(
    "Alias equivalente a ct:Film (owl:equivalentClass tra classi nominate, stile Lezione 12).", lang="it"
)))

# Partizione disgiunta tra tutte le 10 radici di dominio
all_disj = BNode()
g.add((all_disj, RDF.type, OWL.AllDisjointClasses))
members_node = BNode()
Collection(g, members_node, [
    CT.OperaAudiovisiva, CT.Persona, CT.Genere, CT.Tropo,
    CT.Premio, CT.Paese, CT.Lingua, CT.Studio, CT.Collezione, CT.Fonte,
])
g.add((all_disj, OWL.members, members_node))

print("  → classi primitive, disjointUnionOf e AllDisjointClasses aggiunte")

[TBox] Classi primitive, disjointUnionOf e AllDisjointClasses...
  → classi primitive, disjointUnionOf e AllDisjointClasses aggiunte


In [4]:
# ── Cella 2b: Object properties (gerarchie, inverse, simmetria, transitività, chain) ─
print("[TBox] Object properties...")

def add_object_property(uri, label_it, label_en, domain=None, range_=None,
                        characteristics=(), inverse_of=None, subproperty_of=None, comment_it=None):
    g.add((uri, RDF.type, OWL.ObjectProperty))
    g.add((uri, RDFS.label, Literal(label_it, lang="it")))
    g.add((uri, RDFS.label, Literal(label_en, lang="en")))
    if comment_it:
        g.add((uri, RDFS.comment, Literal(comment_it, lang="it")))
    if domain:
        g.add((uri, RDFS.domain, domain))
    if range_:
        g.add((uri, RDFS.range, range_))
    for char in characteristics:
        g.add((uri, RDF.type, char))
    if inverse_of:
        g.add((uri, OWL.inverseOf, inverse_of))
    if subproperty_of:
        g.add((uri, RDFS.subPropertyOf, subproperty_of))
    return uri

# Superproprietà contributori + inverse (pattern Library L11-12: hasPossession / possessedBy)
add_object_property(CT.hasContributor, "ha contributore", "has contributor",
                    CT.OperaAudiovisiva, CT.Persona,
                    comment_it="Superproprietà di directedBy e starring (pattern gerarchia RDFS/OWL Lezione 9 e 11).")
add_object_property(CT.contributedTo, "ha contribuito a", "contributed to",
                    CT.Persona, CT.OperaAudiovisiva, inverse_of=CT.hasContributor,
                    comment_it="Inversa di hasContributor; superproprietà di directorOf e actedIn.")

# Relazioni core film (directedBy e starring sono ruoli semplici: Asymmetric + Irreflexive)
add_object_property(CT.directedBy, "diretto da", "directed by",
                    CT.OperaAudiovisiva, CT.Regista,
                    characteristics=(OWL.AsymmetricProperty, OWL.IrreflexiveProperty),
                    subproperty_of=CT.hasContributor)
add_object_property(CT.directorOf, "ha diretto", "directed",
                    CT.Regista, CT.OperaAudiovisiva,
                    inverse_of=CT.directedBy, subproperty_of=CT.contributedTo)
add_object_property(CT.starring, "interpretato da", "starring",
                    CT.OperaAudiovisiva, CT.Attore,
                    characteristics=(OWL.AsymmetricProperty, OWL.IrreflexiveProperty),
                    subproperty_of=CT.hasContributor)
add_object_property(CT.actedIn, "ha recitato in", "acted in",
                    CT.Attore, CT.OperaAudiovisiva,
                    inverse_of=CT.starring, subproperty_of=CT.contributedTo)

add_object_property(CT.hasGenre, "appartiene al genere", "has genre",
                    CT.OperaAudiovisiva, CT.Genere)
add_object_property(CT.hasTrope, "usa il tropo", "has trope",
                    CT.OperaAudiovisiva, CT.Tropo)
add_object_property(CT.hasAward, "ha ricevuto il premio", "has award",
                    CT.OperaAudiovisiva, CT.Premio,
                    comment_it="Materializzato dalla query Wikidata (premi Oscar Best Picture).")
add_object_property(CT.originCountry, "paese di origine", "origin country",
                    CT.OperaAudiovisiva, CT.Paese)
add_object_property(CT.originalLanguage, "lingua originale", "original language",
                    CT.OperaAudiovisiva, CT.Lingua)
add_object_property(CT.producedBy, "prodotto da", "produced by",
                    CT.OperaAudiovisiva, CT.Studio)
add_object_property(CT.partOfCollection, "parte del franchise", "part of collection",
                    CT.OperaAudiovisiva, CT.Collezione)
add_object_property(CT.hasPart, "comprende", "includes",
                    CT.Collezione, CT.OperaAudiovisiva, inverse_of=CT.partOfCollection)

# Gerarchia dei generi + property chain hasAncestorGenre ≡ hasGenre ∘ subGenreOf
add_object_property(CT.subGenreOf, "sottogenere di", "subgenre of",
                    CT.Genere, CT.Genere, (OWL.TransitiveProperty,),
                    comment_it="Proprietà transitiva sulla tassonomia dei generi (es. ScienceFiction ⊑ SpeculativeFiction).")
add_object_property(CT.hasAncestorGenre, "genere antenato", "has ancestor genre",
                    CT.OperaAudiovisiva, CT.Genere,
                    comment_it="Inferita via property chain hasGenre ∘ subGenreOf.")
chain_genre = BNode()
Collection(g, chain_genre, [CT.hasGenre, CT.subGenreOf])
g.add((CT.hasAncestorGenre, OWL.propertyChainAxiom, chain_genre))

# Sequel: separazione OWL 2 DL tra ruolo semplice (directSequelOf) e non semplice (sequelOf)
add_object_property(CT.sequelOf, "sequel di", "sequel of",
                    CT.OperaAudiovisiva, CT.OperaAudiovisiva, (OWL.TransitiveProperty,),
                    comment_it="Relazione transitiva di sequel (ruolo non semplice in OWL 2 DL).")
add_object_property(CT.hasSequel, "ha come sequel", "has sequel",
                    CT.OperaAudiovisiva, CT.OperaAudiovisiva, (OWL.TransitiveProperty,),
                    inverse_of=CT.sequelOf)
add_object_property(CT.directSequelOf, "sequel diretto di", "direct sequel of",
                    CT.OperaAudiovisiva, CT.OperaAudiovisiva,
                    characteristics=(OWL.AsymmetricProperty, OWL.IrreflexiveProperty, OWL.FunctionalProperty),
                    subproperty_of=CT.sequelOf,
                    comment_it="Ruolo semplice (Asymmetric + Irreflexive + Functional) sottoproprietà del transitivo sequelOf.")

# Collaborazioni (gerarchia simmetrica)
add_object_property(CT.collaboratedWith, "ha collaborato con", "collaborated with",
                    CT.Persona, CT.Persona, (OWL.SymmetricProperty,))
add_object_property(CT.frequentCollaboration, "collaborazione frequente", "frequent collaboration",
                    CT.Persona, CT.Persona, (OWL.SymmetricProperty,),
                    subproperty_of=CT.collaboratedWith)
add_object_property(CT.hasFrequentCollaborator, "ha collaboratore fisso", "has frequent collaborator",
                    CT.Persona, CT.Persona, (OWL.SymmetricProperty,),
                    subproperty_of=CT.collaboratedWith,
                    comment_it="Inferita dalla regola SWRL FrequentCollaboratorRule (diamante regista-attore su ≥ 2 film).")
add_object_property(CT.coStarsWith, "ha recitato insieme a", "co-stars with",
                    CT.Attore, CT.Attore, (OWL.SymmetricProperty,),
                    subproperty_of=CT.collaboratedWith,
                    comment_it="Inferita dalla regola SWRL RecurrentCoStarRule (diamante tra attori su ≥ 2 film).")

# Superproprietà simmetrica tra film correlati
add_object_property(CT.relatedFilm, "film correlato a", "related film",
                    CT.OperaAudiovisiva, CT.OperaAudiovisiva, (OWL.SymmetricProperty,),
                    comment_it="Superproprietà simmetrica di tutte le relazioni di affinità tra opere.")

# Property chain: sameDirectorAs ≡ directedBy ∘ directorOf
add_object_property(CT.sameDirectorAs, "stesso regista di", "same director as",
                    CT.OperaAudiovisiva, CT.OperaAudiovisiva, (OWL.SymmetricProperty,),
                    subproperty_of=CT.relatedFilm,
                    comment_it="Inferita via property chain directedBy ∘ directorOf.")
chain1 = BNode()
Collection(g, chain1, [CT.directedBy, CT.directorOf])
g.add((CT.sameDirectorAs, OWL.propertyChainAxiom, chain1))

# Relazioni di spiegazione e derivazione tra film
for prop, label_it, label_en in [
    (CT.sharesTropeWith, "condivide un tropo con", "shares trope with"),
    (CT.sharesDirectorWith, "condivide il regista con", "shares director with"),
    (CT.sharesActorWith, "condivide un attore con", "shares actor with"),
    (CT.sharesGenreWith, "condivide un genere con", "shares genre with"),
    (CT.sameCollectionAs, "stesso franchise di", "same franchise as"),
    (CT.recommends, "raccomanda", "recommends"),
]:
    add_object_property(prop, label_it, label_en, CT.OperaAudiovisiva, CT.OperaAudiovisiva,
                        (OWL.SymmetricProperty,), subproperty_of=CT.relatedFilm)

# Restrizioni universali necessarie (owl:allValuesFrom) su Film
for prop_uri, rng_cls in [
    (CT.directedBy, CT.Regista),
    (CT.hasGenre, CT.Genere),
    (CT.originCountry, CT.Paese),
]:
    g.add((CT.Film, RDFS.subClassOf, owl_restriction(prop_uri, OWL.allValuesFrom, rng_cls)))

print("  → object properties, gerarchie, property chain e allValuesFrom aggiunte")

[TBox] Object properties...


  → object properties, gerarchie, property chain e allValuesFrom aggiunte


In [5]:
# ── Cella 2c: Data properties, pattern NaviFinal (L10) e owl:hasKey ───────
print("[TBox] Data properties e owl:hasKey...")

def add_data_property(uri, label_it, label_en, domain=None, range_=None,
                      functional=False, subproperty_of=None, comment_it=None):
    g.add((uri, RDF.type, OWL.DatatypeProperty))
    g.add((uri, RDFS.label, Literal(label_it, lang="it")))
    g.add((uri, RDFS.label, Literal(label_en, lang="en")))
    if comment_it:
        g.add((uri, RDFS.comment, Literal(comment_it, lang="it")))
    if domain:
        g.add((uri, RDFS.domain, domain))
    if range_:
        g.add((uri, RDFS.range, range_))
    if functional:
        g.add((uri, RDF.type, OWL.FunctionalProperty))
    if subproperty_of:
        g.add((uri, RDFS.subPropertyOf, subproperty_of))
    return uri

add_data_property(CT.title, "titolo", "title", CT.OperaAudiovisiva, XSD.string, functional=True)
add_data_property(CT.releaseYear, "anno di uscita", "release year",
                  CT.OperaAudiovisiva, XSD.integer, functional=True)
add_data_property(CT.releaseDate, "data di uscita", "release date",
                  CT.OperaAudiovisiva, XSD.string, functional=True)
add_data_property(CT.runtime, "durata (minuti)", "runtime (minutes)",
                  CT.OperaAudiovisiva, XSD.integer, functional=True,
                  comment_it="Proprietà funzionale: consente a HermiT di dimostrare ¬EpicFilm sotto OWA per film con runtime < 150.")
add_data_property(CT.voteAverage, "voto medio", "vote average",
                  CT.OperaAudiovisiva, XSD.decimal, functional=True)
add_data_property(CT.voteCount, "numero di voti", "vote count",
                  CT.OperaAudiovisiva, XSD.integer, functional=True)
add_data_property(CT.budget, "budget (USD)", "budget (USD)",
                  CT.OperaAudiovisiva, XSD.integer, functional=True)
add_data_property(CT.revenue, "incasso (USD)", "revenue (USD)",
                  CT.OperaAudiovisiva, XSD.integer, functional=True)
add_data_property(CT.roi, "ritorno economico (ROI)", "return on investment (ROI)",
                  CT.OperaAudiovisiva, XSD.decimal, functional=True,
                  comment_it="Rapporto revenue/budget (funzionale): abilita ProfitableFilm e BoxOfficeFlop in OWL 2 DL.")
add_data_property(CT.popularity, "popolarità TMDB", "TMDB popularity",
                  CT.OperaAudiovisiva, XSD.decimal, functional=True)
add_data_property(CT.castSize, "dimensione cast", "cast size",
                  CT.OperaAudiovisiva, XSD.integer, functional=True)
add_data_property(CT.awardCode, "codice premio", "award code",
                  CT.OperaAudiovisiva, XSD.string)
add_data_property(CT.overview, "sinossi", "overview", CT.OperaAudiovisiva, XSD.string, functional=True)
add_data_property(CT.tagline, "slogan", "tagline", CT.OperaAudiovisiva, XSD.string)
add_data_property(CT.tmdbId, "TMDB ID", "TMDB ID",
                  CT.OperaAudiovisiva, XSD.integer, functional=True)
add_data_property(CT.imdbId, "IMDb ID", "IMDb ID", CT.OperaAudiovisiva, XSD.string, functional=True)
add_data_property(CT.posterPath, "percorso poster", "poster path", CT.OperaAudiovisiva, XSD.string)
add_data_property(CT.backdropPath, "percorso sfondo", "backdrop path", CT.OperaAudiovisiva, XSD.string)

# Pattern L10 NaviFinal (anti-contaminazione di dominio):
# isoCode è superproprietà neutra SENZA rdfs:domain; Paese e Lingua (disgiunte!)
# usano sottoproprietà con dominio specifico.
add_data_property(CT.isoCode, "codice ISO", "ISO code", range_=XSD.string, functional=True,
                  comment_it="Superproprietà neutra senza rdfs:domain (pattern NaviFinal Lezione 10) per evitare contaminazione tra Paese e Lingua.")
add_data_property(CT.countryIsoCode, "codice ISO paese", "country ISO code",
                  domain=CT.Paese, range_=XSD.string, functional=True, subproperty_of=CT.isoCode)
add_data_property(CT.languageIsoCode, "codice ISO lingua", "language ISO code",
                  domain=CT.Lingua, range_=XSD.string, functional=True, subproperty_of=CT.isoCode)

# Proprietà Persona (DBpedia + TMDB)
add_data_property(CT.birthDate, "data di nascita", "birth date",
                  CT.Persona, XSD.string, functional=True)
add_data_property(CT.birthYear, "anno di nascita", "birth year",
                  CT.Persona, XSD.integer, functional=True,
                  comment_it="Anno estratto da dbo:birthDate (DBpedia) per classificazione OWL 2 DL (VeteranDirector).")
add_data_property(CT.profilePath, "percorso foto", "profile path", CT.Persona, XSD.string)

print("  → data properties e pattern NaviFinal aggiunte")

[TBox] Data properties e owl:hasKey...
  → data properties e pattern NaviFinal aggiunte


In [6]:
# ── Cella 2d: 18 Classi Definite OWL 2 DL (copertura completa SROIQ(D)) ───
print("[TBox] Classi definite OWL 2 DL...")

# ① AcclaimedFilm — datatype restriction (xsd:minInclusive su decimal)
dt_acclaimed = owl_datatype_restriction(XSD.decimal, XSD.minInclusive, Literal("7.5", datatype=XSD.decimal))
add_defined_class(
    CT.AcclaimedFilm, "Film acclamato", "Acclaimed film",
    "Film con voto medio TMDB ≥ 7.5 (datatype restriction xsd:minInclusive). Classificato da HermiT.",
    CT.Film, owl_intersection(CT.Film, owl_restriction(CT.voteAverage, OWL.someValuesFrom, dt_acclaimed))
)

# ② Blockbuster — datatype restriction (xsd:minInclusive su integer)
dt_blockbuster = owl_datatype_restriction(XSD.integer, XSD.minInclusive, Literal(500000000, datatype=XSD.integer))
add_defined_class(
    CT.Blockbuster, "Blockbuster", "Blockbuster",
    "Film con incasso ≥ 500 milioni USD (datatype restriction xsd:minInclusive). Classificato da HermiT.",
    CT.Film, owl_intersection(CT.Film, owl_restriction(CT.revenue, OWL.someValuesFrom, dt_blockbuster))
)

# ③ EpicFilm — datatype restriction (runtime ≥ 150)
dt_epic = owl_datatype_restriction(XSD.integer, XSD.minInclusive, Literal(150, datatype=XSD.integer))
add_defined_class(
    CT.EpicFilm, "Film kolossal", "Epic-length film",
    "Film con durata ≥ 150 minuti (datatype restriction). Classificato da HermiT.",
    CT.Film, owl_intersection(CT.Film, owl_restriction(CT.runtime, OWL.someValuesFrom, dt_epic))
)

# ④ StandardLengthFilm — owl:complementOf + FunctionalProperty (dimostrazione OWA!)
add_defined_class(
    CT.StandardLengthFilm, "Film di durata standard", "Standard-length film",
    "Film ⊓ ¬EpicFilm (owl:complementOf). Poiché runtime è FunctionalProperty, HermiT dimostra "
    "che i film con runtime < 150 appartengono a ¬EpicFilm, mentre lascia NON classificati i film "
    "privi di runtime (es. Insecta) per la Open World Assumption!",
    CT.Film, owl_intersection(CT.Film, owl_complement(CT.EpicFilm))
)

# ⑤ EnsembleFilm — datatype restriction (castSize ≥ 7)
dt_ensemble = owl_datatype_restriction(XSD.integer, XSD.minInclusive, Literal(7, datatype=XSD.integer))
add_defined_class(
    CT.EnsembleFilm, "Film corale", "Ensemble film",
    "Film con almeno 7 attori accreditati (castSize ≥ 7). Classificato da HermiT.",
    CT.Film, owl_intersection(CT.Film, owl_restriction(CT.castSize, OWL.someValuesFrom, dt_ensemble))
)

# ⑥ ProfitableFilm — datatype restriction (roi ≥ 2.0)
dt_prof = owl_datatype_restriction(XSD.decimal, XSD.minInclusive, Literal("2.0", datatype=XSD.decimal))
add_defined_class(
    CT.ProfitableFilm, "Film profittevole", "Profitable film",
    "Film con incasso almeno doppio rispetto al budget (roi ≥ 2.0). Disgiunto da BoxOfficeFlop.",
    CT.Film, owl_intersection(CT.Film, owl_restriction(CT.roi, OWL.someValuesFrom, dt_prof))
)

# ⑦ BoxOfficeFlop — doppia datatype restriction (xsd:minInclusive + xsd:maxExclusive!)
dt_high_budget = owl_datatype_restriction(XSD.integer, XSD.minInclusive, Literal(20000000, datatype=XSD.integer))
dt_low_roi = owl_datatype_restriction(XSD.decimal, [
    (XSD.minExclusive, Literal("0.0", datatype=XSD.decimal)),
    (XSD.maxExclusive, Literal("1.0", datatype=XSD.decimal)),
])
add_defined_class(
    CT.BoxOfficeFlop, "Flop al botteghino", "Box-office flop",
    "Film con budget ≥ 20M USD e 0 < roi < 1.0 (doppia datatype restriction minExclusive/maxExclusive). Disgiunto da ProfitableFilm.",
    CT.Film, owl_intersection(
        CT.Film,
        owl_restriction(CT.budget, OWL.someValuesFrom, dt_high_budget),
        owl_restriction(CT.roi, OWL.someValuesFrom, dt_low_roi),
    )
)
g.add((CT.ProfitableFilm, OWL.disjointWith, CT.BoxOfficeFlop))

# ⑧ CollectionFilm — someValuesFrom su classe nominata
add_defined_class(
    CT.CollectionFilm, "Film di un franchise", "Franchise film",
    "Film che fa parte di una collezione (someValuesFrom su classe nominata). Inferito da HermiT.",
    CT.Film, owl_intersection(CT.Film, owl_restriction(CT.partOfCollection, OWL.someValuesFrom, CT.Collezione))
)

# ⑨ Auteur — someValuesFrom su CLASSE DEFINITA + proprietà inversa (directorOf)!
add_defined_class(
    CT.Auteur, "Regista auteur", "Auteur director",
    "Regista che ha diretto almeno un AcclaimedFilm: Regista ⊓ (directorOf some AcclaimedFilm). "
    "Combina proprietà inversa (directorOf = directedBy⁻¹) e classe definita.",
    CT.Regista, owl_intersection(CT.Regista, owl_restriction(CT.directorOf, OWL.someValuesFrom, CT.AcclaimedFilm))
)

# ⑩ VeteranDirector — datatype restriction su dati DBpedia (birthYear ≤ 1975)
dt_vet = owl_datatype_restriction(XSD.integer, XSD.maxInclusive, Literal(1975, datatype=XSD.integer))
add_defined_class(
    CT.VeteranDirector, "Regista veterano", "Veteran director",
    "Regista nato entro il 1975 (xsd:maxInclusive su birthYear da DBpedia). Classificato da HermiT.",
    CT.Regista, owl_intersection(CT.Regista, owl_restriction(CT.birthYear, OWL.someValuesFrom, dt_vet))
)

# ⑪ VeteranAuteur — intersezione cross-source (TMDB AcclaimedFilm + DBpedia birthYear)
add_defined_class(
    CT.VeteranAuteur, "Auteur veterano", "Veteran auteur",
    "Regista che è sia Auteur (dati TMDB) sia VeteranDirector (dati DBpedia): intersezione cross-source.",
    CT.Auteur, owl_intersection(CT.Auteur, CT.VeteranDirector)
)

# ⑫ MultiGenreFilm — minQualifiedCardinality (+ owl:AllDifferent sui 17 generi)
add_defined_class(
    CT.MultiGenreFilm, "Film multigenere", "Multi-genre film",
    "Film appartenente ad almeno 3 generi distinti (minQualifiedCardinality + owl:AllDifferent sotto OWA).",
    CT.Film, owl_intersection(CT.Film, add_min_qualified_cardinality(CT.hasGenre, 3, CT.Genere))
)

# ⑬ AwardedFilm — owl:hasValue su codice premio Wikidata (Oscar Best Picture)
g.add((CT.OscarBestPicture, RDF.type, CT.Premio))
g.add((CT.OscarBestPicture, RDF.type, OWL.NamedIndividual))
g.add((CT.OscarBestPicture, RDFS.label, Literal("Oscar Best Picture", lang="en")))
g.add((CT.OscarBestPicture, RDFS.label, Literal("Oscar Miglior Film", lang="it")))
g.add((CT.OscarBestPicture, RDFS.comment, Literal(
    "Academy Award for Best Picture (Wikidata wd:Q102427).", lang="it"
)))
g.add((CT.OscarBestPicture, OWL.sameAs, WD.Q102427))
g.add((WD.Q102427, RDF.type, OWL.NamedIndividual))

add_defined_class(
    CT.AwardedFilm, "Film premiato Oscar", "Oscar-awarded film",
    "Film vincitore del premio Oscar per il Miglior Film (owl:hasValue). Dati da Wikidata.",
    CT.Film, owl_intersection(
        CT.Film, owl_restriction(CT.awardCode, OWL.hasValue, Literal("OSCAR_BEST_PICTURE", datatype=XSD.string))
    )
)

# ⑭ AcclaimedBlockbuster — owl:intersectionOf di due classi definite
add_defined_class(
    CT.AcclaimedBlockbuster, "Blockbuster acclamato", "Acclaimed blockbuster",
    "Intersezione di due classi definite: AcclaimedFilm ⊓ Blockbuster (chiusura transitiva HermiT).",
    CT.Film, owl_intersection(CT.AcclaimedFilm, CT.Blockbuster)
)

# ⑮ RegistaAttore — owl:intersectionOf di ruoli (contrasto OWL 2 DL vs SWRL AutoreInterprete!)
add_defined_class(
    CT.RegistaAttore, "Regista e attore", "Director-actor",
    "Persona che è sia Regista sia Attore (Regista ⊓ Attore) anche in film diversi. "
    "Da confrontare con AutoreInterprete (SWRL), che richiede entrambi i ruoli sullo STESSO film.",
    CT.Persona, owl_intersection(CT.Regista, CT.Attore)
)

# ⑯ NotableFilm — owl:unionOf di tre classi definite (contrasto con unione aperta RDFS L9)
add_defined_class(
    CT.NotableFilm, "Film notevole", "Notable film",
    "Unione chiusa di classi definite (owl:unionOf): AcclaimedFilm ⊔ AwardedFilm ⊔ Blockbuster.",
    CT.Film, owl_union(CT.AcclaimedFilm, CT.AwardedFilm, CT.Blockbuster)
)

print("  → 16 classi definite OWL 2 DL aggiunte (ottimizzate per hypertableau HermiT)")

[TBox] Classi definite OWL 2 DL...
  → 16 classi definite OWL 2 DL aggiunte (ottimizzate per hypertableau HermiT)


In [7]:
# ── Cella 2e: Allineamenti a vocabolari esterni (strict OWL 2 DL) ──────────
print("[TBox] Allineamenti schema.org, DBpedia, Wikidata, FOAF...")

class_alignments = [
    (CT.Film, SCHEMA.Movie),
    (CT.Film, DBO.Film),
    (CT.Film, WD.Q11424),
    (CT.SerieTV, SCHEMA.TVSeries),
    (CT.Persona, FOAF.Person),
    (CT.Persona, SCHEMA.Person),
    (CT.Persona, DBO.Person),
    (CT.Regista, DBO.MovieDirector),
    (CT.Attore, DBO.Actor),
    (CT.Paese, SCHEMA.Country),
    (CT.Lingua, SCHEMA.Language),
    (CT.Studio, SCHEMA.Organization),
]
for local_cls, ext_cls in class_alignments:
    g.add((ext_cls, RDF.type, OWL.Class))
    g.add((local_cls, OWL.equivalentClass, ext_cls))

obj_prop_alignments = [
    (CT.directedBy, SCHEMA.director),
    (CT.directedBy, DBO.director),
    (CT.directedBy, WDT.P57),
    (CT.starring, SCHEMA.actor),
    (CT.starring, DBO.starring),
    (CT.starring, WDT.P161),
    (CT.hasGenre, SCHEMA.genre),
    (CT.hasAward, SCHEMA.award),
    (CT.hasAward, WDT.P166),
]
for local_prop, ext_prop in obj_prop_alignments:
    g.add((ext_prop, RDF.type, OWL.ObjectProperty))
    g.add((local_prop, OWL.equivalentProperty, ext_prop))

data_prop_alignments = [
    (CT.releaseDate, SCHEMA.datePublished),
    (CT.birthDate, SCHEMA.birthDate),
    (CT.birthDate, DBO.birthDate),
    (CT.runtime, SCHEMA.duration),
]
for local_prop, ext_prop in data_prop_alignments:
    g.add((ext_prop, RDF.type, OWL.DatatypeProperty))
    g.add((local_prop, OWL.equivalentProperty, ext_prop))

print("  → allineamenti tipizzati OWL 2 DL aggiunti")
print(f"\n[TBox] Triple totali TBox: {len(g)}")

[TBox] Allineamenti schema.org, DBpedia, Wikidata, FOAF...
  → allineamenti tipizzati OWL 2 DL aggiunti

[TBox] Triple totali TBox: 833


## Cella 3 — Fetch Wikidata: premi Oscar Best Picture + link `owl:sameAs`

Interroghiamo l'endpoint SPARQL pubblico di Wikidata (`https://query.wikidata.org/sparql`) tramite `imdb_id` (`wdt:P345`) per identificare i vincitori dell'Oscar al Miglior Film (`wdt:P166 wd:Q102427`) e recuperare i loro URI Wikidata (`wd:Q...`) per creare collegamenti Linked Open Data (`owl:sameAs`). Il risultato è salvato in `data/generated/wikidata_oscar_cache.json`.

In [8]:
# ── Cella 3: Fetch Wikidata (premi Oscar + IRI wd:Q...) ───────────────────
from SPARQLWrapper import SPARQLWrapper, JSON

imdb_to_tmdb = {f.get("imdb_id"): f["id"] for f in top100 if f.get("imdb_id")}
print(f"Film con IMDb ID: {len(imdb_to_tmdb)} / {len(top100)}")

WIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"
cache_path = GENERATED_DIR / "wikidata_oscar_cache.json"
KNOWN_WD_FALLBACK = {
    "tt0075148": "http://www.wikidata.org/entity/Q188652",   # Rocky (1976)
    "tt6751668": "http://www.wikidata.org/entity/Q61448040", # Parasite (2019)
}

wikidata_oscar_winners = set()
wikidata_film_iris = {}

try:
    sparql = SPARQLWrapper(WIKIDATA_ENDPOINT)
    sparql.setTimeout(25)
    sparql.addCustomHttpHeader(
        "User-Agent",
        "CineTrope/3.1 (https://github.com/cinetrope; university semantic web project)",
    )
    imdb_ids_values = " ".join(f'"{iid}"' for iid in sorted(imdb_to_tmdb))
    query = f"""
    SELECT DISTINCT ?imdb ?film WHERE {{
      VALUES ?imdb {{ {imdb_ids_values} }}
      ?film wdt:P345 ?imdb .
      ?film wdt:P166 wd:Q102427 .
    }}
    """
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    for row in results["results"]["bindings"]:
        iid = row["imdb"]["value"]
        wikidata_oscar_winners.add(iid)
        if "film" in row:
            wikidata_film_iris[iid] = row["film"]["value"]
    print(f"✓ Wikidata: {len(wikidata_oscar_winners)} film vincitori Oscar Best Picture nel corpus")
    for iid in sorted(wikidata_oscar_winners):
        title = next((f["title"] for f in top100 if f.get("imdb_id") == iid), iid)
        print(f"    • {title} ({iid}) → {wikidata_film_iris.get(iid, '?')}")
    cache_path.write_text(
        json.dumps(
            {"imdb_winners": sorted(wikidata_oscar_winners), "imdb_to_wd": wikidata_film_iris},
            indent=2,
        ),
        encoding="utf-8",
    )
    print(f"  Cache salvata in {cache_path}")
except Exception as e:
    print(f"⚠ Wikidata non raggiungibile: {e}")
    if cache_path.exists():
        cached = json.loads(cache_path.read_text(encoding="utf-8"))
        wikidata_oscar_winners = set(cached.get("imdb_winners", []))
        wikidata_film_iris = dict(cached.get("imdb_to_wd", {}))
        for iid in wikidata_oscar_winners:
            if iid not in wikidata_film_iris and iid in KNOWN_WD_FALLBACK:
                wikidata_film_iris[iid] = KNOWN_WD_FALLBACK[iid]
        print(f"  Caricati {len(wikidata_oscar_winners)} vincitori dalla cache locale ({len(wikidata_film_iris)} IRI wd:).")
    else:
        print("  Nessuna cache locale trovata (AwardedFilm resterà vuota).")

Film con IMDb ID: 100 / 100


✓ Wikidata: 2 film vincitori Oscar Best Picture nel corpus
    • Rocky (tt0075148) → http://www.wikidata.org/entity/Q188652
    • Parasite (tt6751668) → http://www.wikidata.org/entity/Q61448040
  Cache salvata in C:\uni\CineTrope\data\generated\wikidata_oscar_cache.json


## Cella 4 — Fetch DBpedia: date di nascita registi + link `owl:sameAs`

Interroghiamo DBpedia (`https://dbpedia.org/sparql`) a blocchi (`VALUES ?p { <http://dbpedia.org/resource/...> }`) per recuperare `dbo:birthDate` dei registi del corpus e conservare gli URI `dbr:` per creare collegamenti **Linked Open Data (`owl:sameAs`)**. I risultati vengono salvati nella cache `data/generated/dbpedia_directors_cache.json` così da garantire riproducibilità anche offline.

In [9]:
# ── Cella 4: Fetch DBpedia (date di nascita + IRI dbr:) ───────────────────
DBPEDIA_ENDPOINT = "https://dbpedia.org/sparql"
directors_in_corpus = sorted({d for f in top100 for d in f.get("directors", [])})
print(f"Registi nel corpus: {len(directors_in_corpus)}")

def _dbpedia_iri(name: str) -> str:
    return f"http://dbpedia.org/resource/{quote(name.replace(' ', '_'), safe='-_.~')}"

dbpedia_birthdates = {}
dbpedia_iris = {}

# 1. Carica cache locale (people.json + dbpedia_birthdates.json + dbpedia_directors_cache.json)
for path in (RAW_DIR / "people.json", RAW_DIR / "dbpedia_birthdates.json"):
    if not path.exists():
        continue
    raw_data = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(raw_data, dict):
        for name, info in raw_data.items():
            if isinstance(info, dict) and info.get("birthDate"):
                dbpedia_birthdates[name] = str(info["birthDate"])[:10]
            elif isinstance(info, str) and info:
                dbpedia_birthdates[name] = info[:10]
    elif isinstance(raw_data, list):
        for r in raw_data:
            if r.get("name") and r.get("birthDate"):
                dbpedia_birthdates[r["name"]] = str(r["birthDate"])[:10]

dbp_gen_cache = GENERATED_DIR / "dbpedia_directors_cache.json"
if dbp_gen_cache.exists():
    cached_dbp = json.loads(dbp_gen_cache.read_text(encoding="utf-8"))
    for name, rec in cached_dbp.items():
        if rec.get("birthDate"):
            dbpedia_birthdates[name] = str(rec["birthDate"])[:10]
        if rec.get("iri"):
            dbpedia_iris[name] = rec["iri"]

# 2. Query DBpedia per i registi mancanti dalla cache (o senza IRI confermato)
missing_directors = [d for d in directors_in_corpus if d not in dbpedia_birthdates or d not in dbpedia_iris]
print(f"  Date già in cache locale: {sum(1 for d in directors_in_corpus if d in dbpedia_birthdates)}/{len(directors_in_corpus)}")
print(f"  Registi da verificare su DBpedia: {len(missing_directors)}")

dbpedia_new = {}
if missing_directors:
    try:
        sparql_dbp = SPARQLWrapper(DBPEDIA_ENDPOINT)
        sparql_dbp.setTimeout(15)
        sparql_dbp.addCustomHttpHeader("User-Agent", "CineTrope/3.1 (university project)")
        sparql_dbp.setReturnFormat(JSON)
        chunk_size = 25
        for i in range(0, len(missing_directors), chunk_size):
            chunk = missing_directors[i : i + chunk_size]
            iri_to_name = {_dbpedia_iri(d): d for d in chunk}
            values = " ".join(f"<{iri}>" for iri in iri_to_name)
            q = f"""
            PREFIX dbo: <http://dbpedia.org/ontology/>
            SELECT ?p ?bd WHERE {{
              VALUES ?p {{ {values} }}
              ?p dbo:birthDate ?bd .
            }}
            """
            sparql_dbp.setQuery(q)
            try:
                res = sparql_dbp.query().convert()
                for row in res["results"]["bindings"]:
                    iri_val = row["p"]["value"]
                    bd_val = row["bd"]["value"][:10]
                    orig = iri_to_name.get(iri_val)
                    if orig and len(bd_val) == 10 and bd_val[4] == "-" and bd_val[7] == "-":
                        dbpedia_new.setdefault(orig, bd_val)
                        dbpedia_iris[orig] = iri_val
            except Exception as chunk_err:
                print(f"  ⚠ Blocco {i//chunk_size + 1} DBpedia fallito: {chunk_err}")
                if i == 0:
                    break
        print(f"  → {len(dbpedia_new)} date di nascita confermate da DBpedia")
    except Exception as e:
        print(f"⚠ DBpedia non raggiungibile: {e}")

all_birthdates = {**dbpedia_birthdates, **dbpedia_new}
# Salva cache arricchita per futuri run offline
cache_payload = {
    d: {"birthDate": all_birthdates[d], "iri": dbpedia_iris.get(d, _dbpedia_iri(d))}
    for d in directors_in_corpus
    if d in all_birthdates
}
if cache_payload:
    dbp_gen_cache.write_text(json.dumps(cache_payload, indent=2, ensure_ascii=False), encoding="utf-8")

covered = sum(1 for d in directors_in_corpus if d in all_birthdates)
print(f"  Copertura finale date di nascita + IRI DBpedia: {covered}/{len(directors_in_corpus)}")

Registi nel corpus: 117
  Date già in cache locale: 43/117
  Registi da verificare su DBpedia: 74


  → 0 date di nascita confermate da DBpedia
  Copertura finale date di nascita + IRI DBpedia: 43/117


## Cella 5 — Popolamento ABox (`owl:NamedIndividual`, `owl:sameAs`, `SKOS`, `directSequelOf`, `owl:AllDifferent`)

Popoliamo l'**ABox** con tutti gli individui del corpus top-100:
- Ogni individuo è tipizzato esplicitamente come **`owl:NamedIndividual`** (formato nativo di Protégé, come in `Frutta.owl` / `Researcher.rdf` della Lezione 12).
- Popoliamo anche **`ct:Lingua` / `ct:originalLanguage`** (`ct:languageIsoCode`) e **`ct:countryIsoCode`** per dimostrare il pattern **L10 `NaviFinal`** (`isoCode` senza contaminazione di dominio).
- Colleghiamo con **`owl:sameAs`** i registi trovati su DBpedia (`dbr:`), i film Oscar su Wikidata (`wd:`), e l'alias di genere `ct:SciFi owl:sameAs ct:ScienceFiction` (escluso da `AllDifferent` per evitare contraddizioni!).
- Colleghiamo le catene di sequel all'interno dello stesso franchise tramite **`ct:directSequelOf`** (es. *Avengers: Doomsday* → *Avengers: Endgame* → *Avengers: Infinity War*; *Godzilla Minus Zero* → *Godzilla Minus One*), così HermiT inferisce automaticamente la chiusura transitiva `sequelOf` e l'inversa `hasSequel`.
- Dichiariamo **`owl:AllDifferent`** in modo mirato sotto Open World Assumption per mantenere il tableau rapido:
  1. i **17 generi canonici** (`MultiGenreFilm`, `VersatileAuteurRule`);
  2. i **13 film** che condividono un regista o un franchise (`FrequentCollaboratorRule`, `SharedCollectionRule`, `VersatileAuteurRule` — 78 coppie invece di 4.950!).
- Eseguiamo infine una query **SPARQL `CONSTRUCT` (stile Lezione 8)** per derivare le coppie di co-star ricorrenti (≥ 2 film insieme) in una sola direzione (`a1 ct:coStarsWith a2`), lasciando a **HermiT** l'inferenza della direzione inversa tramite **`owl:SymmetricProperty`** (stile Lezione 11 `ShakespeareSimm`)!

In [10]:
# ── Cella 5: Popolamento ABox ─────────────────────────────────────────────
print("[ABox] Popolamento individui...")

genres_data = json.loads((RAW_DIR / "genres.json").read_text(encoding="utf-8"))

def slugify(value: str) -> str:
    n = unicodedata.normalize("NFKD", value)
    ascii_ = n.encode("ascii", "ignore").decode("ascii")
    return "".join(ch for ch in ascii_ if ch.isalnum()) or "unknown"

def normalize_genre(gid):
    return "ScienceFiction" if gid == "SciFi" else gid

def add_all_different(nodes):
    if len(nodes) < 2:
        return
    diff = BNode()
    g.add((diff, RDF.type, OWL.AllDifferent))
    diff_list = BNode()
    Collection(g, diff_list, sorted(nodes, key=str))
    g.add((diff, OWL.distinctMembers, diff_list))

LANG_NAMES = {
    "en": ("Inglese", "English"),
    "es": ("Spagnolo", "Spanish"),
    "ja": ("Giapponese", "Japanese"),
    "ko": ("Coreano", "Korean"),
    "ta": ("Tamil", "Tamil"),
    "id": ("Indonesiano", "Indonesian"),
    "fr": ("Francese", "French"),
    "it": ("Italiano", "Italian"),
    "de": ("Tedesco", "German"),
    "zh": ("Cinese", "Chinese"),
    "hi": ("Hindi", "Hindi"),
}

# Fonti (named graph)
for src, label in [
    (CT.Ontology, "Ontologia"), (CT.TMDB, "TMDB"), (CT.Wikidata, "Wikidata"),
    (CT.DBpedia, "DBpedia"), (CT.Derived, "Dati derivati"),
    (CT.Inferred, "Inferenze"),
]:
    g.add((src, RDF.type, CT.Fonte))
    g.add((src, RDF.type, OWL.NamedIndividual))
    g.add((src, RDFS.label, Literal(label)))

# ─── Generi (+ SKOS + alias owl:sameAs SciFi ↔ ScienceFiction) ──────────
genre_nodes = set()
for genre in genres_data:
    if genre.get("equivalentTo"):
        alias_node = CT[genre["id"]]
        target_node = CT[normalize_genre(genre["equivalentTo"])]
        g.add((alias_node, RDF.type, CT.Genere))
        g.add((alias_node, RDF.type, OWL.NamedIndividual))
        g.add((alias_node, RDFS.label, Literal(genre["label"], lang="en")))
        g.add((alias_node, SKOS.altLabel, Literal(genre["label"], lang="en")))
        g.add((alias_node, OWL.sameAs, target_node))
        continue
    gid = CT[normalize_genre(genre["id"])]
    genre_nodes.add(gid)
    g.add((gid, RDF.type, CT.Genere))
    g.add((gid, RDF.type, OWL.NamedIndividual))
    g.add((gid, RDFS.label, Literal(genre["label"], lang="it")))
    g.add((gid, SKOS.prefLabel, Literal(genre["label"], lang="it")))
    for parent in genre.get("parents", []):
        parent_node = CT[normalize_genre(parent)]
        genre_nodes.add(parent_node)
        g.add((gid, CT.subGenreOf, parent_node))

add_all_different(genre_nodes)

# ─── Film + Persone + Lingue + Paesi + Studi + Collezioni ────────────────
people_seen = {}
actor_film_counts = Counter()
film_nodes = set()
collection_films = defaultdict(list)  # collection_uri -> [(year, film_uri)]

for film in tqdm(top100, desc="Film ABox"):
    work = CT[film["id"]]
    film_nodes.add(work)
    g.add((work, RDF.type, CT.Film))
    g.add((work, RDF.type, OWL.NamedIndividual))
    g.add((work, RDFS.label, Literal(film["title"], lang="it")))
    g.add((work, CT.displayTitle, Literal(film["title"], lang="it")))
    g.add((work, CT.title, Literal(film["title"], datatype=XSD.string)))

    if film.get("year"):
        g.add((work, CT.releaseYear, Literal(int(film["year"]), datatype=XSD.integer)))
    if film.get("release_date"):
        g.add((work, CT.releaseDate, Literal(str(film["release_date"]), datatype=XSD.string)))
    if film.get("runtime") and film["runtime"] > 0:
        g.add((work, CT.runtime, Literal(int(film["runtime"]), datatype=XSD.integer)))
    if film.get("vote_average") and film["vote_average"] > 0:
        g.add((work, CT.voteAverage, Literal(str(film["vote_average"]), datatype=XSD.decimal)))
    if film.get("vote_count") and film["vote_count"] > 0:
        g.add((work, CT.voteCount, Literal(int(film["vote_count"]), datatype=XSD.integer)))
    budget = film.get("budget") or 0
    revenue = film.get("revenue") or 0
    if budget > 0:
        g.add((work, CT.budget, Literal(int(budget), datatype=XSD.integer)))
    if revenue > 0:
        g.add((work, CT.revenue, Literal(int(revenue), datatype=XSD.integer)))
    if budget > 0 and revenue > 0:
        roi_val = Decimal(revenue) / Decimal(budget)
        g.add((work, CT.roi, Literal(f"{roi_val:.3f}", datatype=XSD.decimal)))
    if film.get("popularity") and film["popularity"] > 0:
        g.add((work, CT.popularity, Literal(str(film["popularity"]), datatype=XSD.decimal)))
    cast_top = film.get("cast", [])[:10]
    if cast_top:
        g.add((work, CT.castSize, Literal(len(cast_top), datatype=XSD.integer)))
    if film.get("overview"):
        g.add((work, CT.overview, Literal(film["overview"], datatype=XSD.string)))
    if film.get("tagline"):
        g.add((work, CT.tagline, Literal(film["tagline"], datatype=XSD.string)))
    if film.get("imdb_id"):
        g.add((work, CT.imdbId, Literal(film["imdb_id"], datatype=XSD.string)))
    if film.get("tmdb_id"):
        g.add((work, CT.tmdbId, Literal(int(film["tmdb_id"]), datatype=XSD.integer)))
    if film.get("poster_path"):
        g.add((work, CT.posterPath, Literal(film["poster_path"], datatype=XSD.string)))
    if film.get("backdrop_path"):
        g.add((work, CT.backdropPath, Literal(film["backdrop_path"], datatype=XSD.string)))

    # Generi
    for gid in film.get("genres", []):
        g.add((work, CT.hasGenre, CT[normalize_genre(gid)]))

    # Tropi / keyword
    for kw in film.get("keywords", []):
        kw_node = CT[kw["id"]]
        g.add((kw_node, RDF.type, CT.Tropo))
        g.add((kw_node, RDF.type, OWL.NamedIndividual))
        g.add((kw_node, RDFS.label, Literal(kw["label"], lang="en")))
        g.add((kw_node, SKOS.prefLabel, Literal(kw["label"], lang="en")))
        g.add((work, CT.hasTrope, kw_node))

    # Registi (+ DBpedia birthDate/birthYear + owl:sameAs dbr:)
    for director in film.get("directors", []):
        d_local = slugify(director)
        d_node = CT[d_local]
        g.add((work, CT.directedBy, d_node))
        g.add((d_node, RDF.type, CT.Persona))
        g.add((d_node, RDF.type, CT.Regista))
        g.add((d_node, RDF.type, OWL.NamedIndividual))
        g.add((d_node, RDFS.label, Literal(director)))
        if director in all_birthdates and all_birthdates[director]:
            bd = str(all_birthdates[director])[:10]
            g.add((d_node, CT.birthDate, Literal(bd, datatype=XSD.string)))
            if bd[:4].isdigit():
                g.add((d_node, CT.birthYear, Literal(int(bd[:4]), datatype=XSD.integer)))
            dbp_uri = URIRef(dbpedia_iris.get(director, _dbpedia_iri(director)))
            g.add((dbp_uri, RDF.type, OWL.NamedIndividual))
            g.add((d_node, OWL.sameAs, dbp_uri))
        people_seen.setdefault(d_local, set()).add("Regista")

    # Attori (top 10)
    for actor in cast_top:
        a_local = slugify(actor)
        a_node = CT[a_local]
        g.add((work, CT.starring, a_node))
        g.add((a_node, RDF.type, CT.Persona))
        g.add((a_node, RDF.type, CT.Attore))
        g.add((a_node, RDF.type, OWL.NamedIndividual))
        g.add((a_node, RDFS.label, Literal(actor)))
        actor_film_counts[a_node] += 1
        people_seen.setdefault(a_local, set()).add("Attore")

    # Lingua originale (usa ct:languageIsoCode ⊑ ct:isoCode — pattern L10 NaviFinal)
    orig_lang = film.get("original_language")
    if orig_lang:
        l_node = CT["Lang_" + orig_lang]
        g.add((l_node, RDF.type, CT.Lingua))
        g.add((l_node, RDF.type, OWL.NamedIndividual))
        g.add((l_node, CT.languageIsoCode, Literal(orig_lang, datatype=XSD.string)))
        lbl_it, lbl_en = LANG_NAMES.get(orig_lang, (orig_lang.upper(), orig_lang.upper()))
        g.add((l_node, RDFS.label, Literal(lbl_it, lang="it")))
        g.add((l_node, RDFS.label, Literal(lbl_en, lang="en")))
        g.add((work, CT.originalLanguage, l_node))

    # Paesi di produzione (usa ct:countryIsoCode ⊑ ct:isoCode — pattern L10 NaviFinal)
    for country in film.get("countries", []):
        c_node = CT["Country_" + country["iso"]]
        g.add((c_node, RDF.type, CT.Paese))
        g.add((c_node, RDF.type, OWL.NamedIndividual))
        g.add((c_node, CT.countryIsoCode, Literal(country["iso"], datatype=XSD.string)))
        if country.get("name"):
            g.add((c_node, RDFS.label, Literal(country["name"], lang="en")))
        g.add((work, CT.originCountry, c_node))

    # Studio di produzione (primi 3)
    for company in film.get("companies", [])[:3]:
        s_node = CT["Studio" + str(company["id"])]
        g.add((s_node, RDF.type, CT.Studio))
        g.add((s_node, RDF.type, OWL.NamedIndividual))
        g.add((s_node, RDFS.label, Literal(company["name"])))
        g.add((work, CT.producedBy, s_node))

    # Franchise / collection
    if film.get("collection"):
        coll = film["collection"]
        c_node = CT["Collection" + str(coll["id"])]
        g.add((c_node, RDF.type, CT.Collezione))
        g.add((c_node, RDF.type, OWL.NamedIndividual))
        g.add((c_node, RDFS.label, Literal(coll["name"])))
        g.add((work, CT.partOfCollection, c_node))
        if film.get("year"):
            collection_films[c_node].append((int(film["year"]), work))

    # Premi Oscar (da Wikidata + owl:sameAs wd:Q...)
    iid = film.get("imdb_id")
    if iid in wikidata_oscar_winners:
        g.add((work, CT.hasAward, CT.OscarBestPicture))
        g.add((work, CT.awardCode, Literal("OSCAR_BEST_PICTURE", datatype=XSD.string)))
        if iid in wikidata_film_iris:
            wd_uri = URIRef(wikidata_film_iris[iid])
            g.add((wd_uri, RDF.type, OWL.NamedIndividual))
            g.add((work, OWL.sameAs, wd_uri))

# ─── Catene di sequel diretti (directSequelOf ⊑ sequelOf transitiva) ─────
sequel_edges = 0
for coll_uri, items in collection_films.items():
    items_sorted = sorted(items, key=lambda x: x[0])
    for (y_prev, f_prev), (y_curr, f_curr) in zip(items_sorted, items_sorted[1:]):
        if y_curr > y_prev:
            g.add((f_curr, CT.directSequelOf, f_prev))
            sequel_edges += 1

# ─── AllDifferent mirato sui soli film che condividono regista o franchise ─
dir_counts = Counter(d for f in top100 for d in f.get("directors", []))
coll_counts = Counter(f["collection"]["id"] for f in top100 if f.get("collection"))
shared_context_films = {
    CT[f["id"]]
    for f in top100
    if any(dir_counts[d] >= 2 for d in f.get("directors", []))
    or (f.get("collection") and coll_counts[f["collection"]["id"]] >= 2)
}
add_all_different(shared_context_films)

# ─── SPARQL CONSTRUCT (Lezione 8) per co-star ricorrenti (≥ 2 film) ──────
# Materializza SOLO una direzione (STR(?a1) < STR(?a2)): HermiT inferisce la
# direzione inversa tramite owl:SymmetricProperty (stile Lezione 11 ShakespeareSimm).
q_costars = """
PREFIX ct: <http://example.org/cinetrope#>
CONSTRUCT { ?a1 ct:coStarsWith ?a2 . }
WHERE {
  SELECT ?a1 ?a2 WHERE {
    ?f ct:starring ?a1, ?a2 .
    FILTER(STR(?a1) < STR(?a2))
  }
  GROUP BY ?a1 ?a2
  HAVING (COUNT(DISTINCT ?f) >= 2)
}
"""
costar_graph = g.query(q_costars).graph
costar_edges = 0
for triple in costar_graph:
    g.add(triple)
    costar_edges += 1

same_as_count = len(list(g.triples((None, OWL.sameAs, None))))
print(f"\n[ABox] Statistiche:")
print(f"  Film totali:                     {len(film_nodes)} (AllDifferent mirato su {len(shared_context_films)} film condivisi)")
print(f"  Persone:                         {len(people_seen)}")
print(f"  Generi canonici (AllDifferent):  {len(genre_nodes)} (+ alias SciFi owl:sameAs ScienceFiction)")
print(f"  Archi directSequelOf:            {sequel_edges}")
print(f"  Coppie coStarsWith (CONSTRUCT):  {costar_edges} (unidirezionali; HermiT inferirà la simmetria)")
print(f"  Collegamenti owl:sameAs (LOD):   {same_as_count}")
print(f"  Triple totali (TBox + ABox):     {len(g)}")

[ABox] Popolamento individui...


Film ABox:   0%|          | 0/100 [00:00<?, ?it/s]


[ABox] Statistiche:
  Film totali:                     100 (AllDifferent mirato su 13 film condivisi)
  Persone:                         1019
  Generi canonici (AllDifferent):  17 (+ alias SciFi owl:sameAs ScienceFiction)
  Archi directSequelOf:            3
  Coppie coStarsWith (CONSTRUCT):  54 (unidirezionali; HermiT inferirà la simmetria)
  Collegamenti owl:sameAs (LOD):   47
  Triple totali (TBox + ABox):     14604


## Cella 6 — 6 Regole SWRL DL-safe (oltre l'espressività di OWL 2 DL)

A differenza di una versione ingenua in cui le regole SWRL replicano costrutti già esprimibili in OWL 2 DL, qui progettiamo **6 regole SWRL DL-safe** eseguite nativamente da **HermiT** (e visibili/abilitate in **Protégé SWRLTab**) che dimostrano il confine reale tra $\mathcal{SROIQ}(\mathcal{D})$ (proprietà del modello ad albero) e le regole di Horn:

| # | Regola | Formula | Perché supera OWL 2 DL |
|---|---|---|---|
| 1 | `SelfDirectedRoleRule` | `Film(?f) ∧ directedBy(?f, ?p) ∧ starring(?f, ?p) → AutoreInterprete(?p)` | **Congiunzione di ruoli** (ciclo a 2 archi paralleli sullo stesso film) — impossibile in $\mathcal{SROIQ}$ |
| 2 | `FrequentCollaboratorRule` | `Film(?f1) ∧ Film(?f2) ∧ differentFrom(?f1,?f2) ∧ directedBy(?f1,?d) ∧ directedBy(?f2,?d) ∧ starring(?f1,?a) ∧ starring(?f2,?a) → hasFrequentCollaborator(?d, ?a)` | **Ciclo a diamante a 4 nodi** — viola la *tree-model property* di OWL 2 DL |
| 3 | `SequelDirectorRule` | `Film(?f1) ∧ Film(?f2) ∧ sequelOf(?f2,?f1) ∧ directedBy(?f1,?d) ∧ directedBy(?f2,?d) → SequelDirector(?d)` | **Triangolo** chiuso sulla proprietà transitiva `sequelOf` — impossibile in OWL 2 DL |
| 4 | `VersatileAuteurRule` | `Regista(?d) ∧ directorOf(?d,?f1) ∧ AcclaimedFilm(?f1) ∧ hasGenre(?f1,?g1) ∧ directorOf(?d,?f2) ∧ hasGenre(?f2,?g2) ∧ differentFrom(?f1,?f2) ∧ differentFrom(?g1,?g2) → VersatileAuteur(?d)` | **Join multi-hop** su film distinti e generi distinti + chaining su `AcclaimedFilm` |
| 5 | `SharedCollectionRule` | `Film(?f1) ∧ Film(?f2) ∧ differentFrom(?f1,?f2) ∧ partOfCollection(?f1,?c) ∧ partOfCollection(?f2,?c) → sameCollectionAs(?f1, ?f2)` | Esclude i self-loop tramite `DifferentIndividualsAtom` (una property chain li includerebbe!) |
| 6 | `EnsembleAwardRule` | `Film(?f) ∧ EnsembleFilm(?f) ∧ AcclaimedFilm(?f) ∧ ProfitableFilm(?f) → PremioFilm(?f)` | **Chaining** su 3 classi definite OWL 2 DL |

> **Nota tecnica su `swrlb:` e HermiT**: HermiT (sia in OWLReady2 sia in Protégé) implementa SWRL DL-safe senza *built-in* aritmetici (`swrlb:*`, che richiedono Drools/Pellet e causerebbero `IllegalArgumentException` all'avvio di HermiT). Per questo l'aritmetica sul rapporto incasso/budget (`roi`) e sull'anno di nascita DBpedia (`birthYear`) è modellata tramite **OWL 2 Datatype Restrictions** (`ProfitableFilm`, `BoxOfficeFlop`, `VeteranDirector`), lasciando a SWRL i **pattern grafici non ad albero** supportati al 100% da HermiT!

In [11]:
# ── Cella 6: 6 Regole SWRL DL-safe (compatibili HermiT + Protégé SWRLTab) ─
print("[SWRL] Aggiunta di 6 regole DL-safe...")

# Classi inferite dalle regole SWRL
add_class(
    CT.AutoreInterprete, "Autore-interprete (stesso film)", "Self-directed actor",
    "Persona che ha diretto E interpretato lo STESSO film (congiunzione di ruoli via SWRL SelfDirectedRoleRule, non esprimibile in OWL 2 DL).",
    subclass_of=CT.RegistaAttore,
)
add_class(
    CT.SequelDirector, "Regista di saga/sequel", "Sequel director",
    "Regista che ha diretto sia un film sia un suo sequel transitivo (triangolo SWRL SequelDirectorRule).",
    subclass_of=CT.Regista,
)
add_class(
    CT.VersatileAuteur, "Auteur versatile", "Versatile auteur",
    "Regista autore di almeno un film acclamato che ha diretto due film distinti di generi distinti (SWRL VersatileAuteurRule).",
    subclass_of=CT.Auteur,
)
add_class(
    CT.PremioFilm, "Film corale acclamato e profittevole", "Acclaimed profitable ensemble film",
    "Film che soddisfa congiuntamente EnsembleFilm, AcclaimedFilm e ProfitableFilm (chaining SWRL EnsembleAwardRule).",
    subclass_of=CT.Film,
)

swrl_g = Graph()
for prefix, ns in [("ct", CT), ("swrl", SWRL), ("swrla", SWRLA), ("rdf", RDF), ("rdfs", RDFS), ("xsd", XSD)]:
    swrl_g.bind(prefix, ns, override=True, replace=True)


def _add_atom_list(target_graph, atom_nodes):
    if not atom_nodes:
        return RDF.nil
    head = BNode()
    curr = head
    for idx, atom in enumerate(atom_nodes):
        target_graph.add((curr, RDF.type, SWRL.AtomList))
        target_graph.add((curr, RDF.first, atom))
        if idx + 1 < len(atom_nodes):
            nxt = BNode()
            target_graph.add((curr, RDF.rest, nxt))
            curr = nxt
        else:
            target_graph.add((curr, RDF.rest, RDF.nil))
    return head


def make_swrl_rule(rule_id, comment_it, body_atoms, head_atoms):
    """Aggiunge una regola SWRL DL-safe con annotazioni Protégé SWRLTab."""
    for target in (g, swrl_g):
        rule = CT[rule_id]
        target.add((rule, RDF.type, SWRL.Imp))
        target.add((rule, SWRLA.isRuleEnabled, Literal(True, datatype=XSD.boolean)))
        target.add((rule, RDFS.label, Literal(rule_id, datatype=XSD.string)))
        target.add((rule, RDFS.comment, Literal(comment_it, lang="it")))

        body_nodes = []
        for atom_type, props in body_atoms:
            atom = BNode()
            target.add((atom, RDF.type, atom_type))
            for p, v in props:
                target.add((atom, p, v))
            body_nodes.append(atom)
        target.add((rule, SWRL.body, _add_atom_list(target, body_nodes)))

        head_nodes = []
        for atom_type, props in head_atoms:
            atom = BNode()
            target.add((atom, RDF.type, atom_type))
            for p, v in props:
                target.add((atom, p, v))
            head_nodes.append(atom)
        target.add((rule, SWRL.head, _add_atom_list(target, head_nodes)))


# Variabili SWRL tipizzate
VAR_F  = URIRef("urn:swrl:var#f")
VAR_F1 = URIRef("urn:swrl:var#f1")
VAR_F2 = URIRef("urn:swrl:var#f2")
VAR_P  = URIRef("urn:swrl:var#p")
VAR_D  = URIRef("urn:swrl:var#d")
VAR_A  = URIRef("urn:swrl:var#a")
VAR_A1 = URIRef("urn:swrl:var#a1")
VAR_A2 = URIRef("urn:swrl:var#a2")
VAR_C  = URIRef("urn:swrl:var#c")
VAR_G1 = URIRef("urn:swrl:var#g1")
VAR_G2 = URIRef("urn:swrl:var#g2")
for var in (VAR_F, VAR_F1, VAR_F2, VAR_P, VAR_D, VAR_A, VAR_A1, VAR_A2, VAR_C, VAR_G1, VAR_G2):
    g.add((var, RDF.type, SWRL.Variable))
    swrl_g.add((var, RDF.type, SWRL.Variable))

def cls_atom(cls_uri, var):
    return (SWRL.ClassAtom, [(SWRL.classPredicate, cls_uri), (SWRL.argument1, var)])

def prop_atom(prop_uri, v1, v2):
    return (SWRL.IndividualPropertyAtom, [(SWRL.propertyPredicate, prop_uri), (SWRL.argument1, v1), (SWRL.argument2, v2)])

def diff_atom(v1, v2):
    return (SWRL.DifferentIndividualsAtom, [(SWRL.argument1, v1), (SWRL.argument2, v2)])

# 1. SelfDirectedRoleRule (congiunzione di ruoli — impossibile in OWL 2 DL)
make_swrl_rule(
    "SelfDirectedRoleRule",
    "Persona che dirige e interpreta lo stesso film → AutoreInterprete (role conjunction non esprimibile in OWL 2 DL).",
    body_atoms=[cls_atom(CT.Film, VAR_F), prop_atom(CT.directedBy, VAR_F, VAR_P), prop_atom(CT.starring, VAR_F, VAR_P)],
    head_atoms=[cls_atom(CT.AutoreInterprete, VAR_P)],
)

# 2. FrequentCollaboratorRule (diamante a 4 nodi — impossibile in OWL 2 DL)
make_swrl_rule(
    "FrequentCollaboratorRule",
    "Regista e attore che collaborano su almeno 2 film distinti → hasFrequentCollaborator (ciclo a diamante non esprimibile in OWL 2 DL).",
    body_atoms=[
        cls_atom(CT.Film, VAR_F1), cls_atom(CT.Film, VAR_F2), diff_atom(VAR_F1, VAR_F2),
        prop_atom(CT.directedBy, VAR_F1, VAR_D), prop_atom(CT.directedBy, VAR_F2, VAR_D),
        prop_atom(CT.starring, VAR_F1, VAR_A), prop_atom(CT.starring, VAR_F2, VAR_A),
    ],
    head_atoms=[prop_atom(CT.hasFrequentCollaborator, VAR_D, VAR_A)],
)

# 3. SequelDirectorRule (triangolo sulla proprietà transitiva sequelOf)
make_swrl_rule(
    "SequelDirectorRule",
    "Regista che dirige sia un film sia un suo sequel (anche transitivo) → SequelDirector (triangolo non esprimibile in OWL 2 DL).",
    body_atoms=[
        cls_atom(CT.Film, VAR_F1), cls_atom(CT.Film, VAR_F2),
        prop_atom(CT.sequelOf, VAR_F2, VAR_F1),
        prop_atom(CT.directedBy, VAR_F1, VAR_D), prop_atom(CT.directedBy, VAR_F2, VAR_D),
    ],
    head_atoms=[cls_atom(CT.SequelDirector, VAR_D)],
)

# 4. VersatileAuteurRule (join multi-hop su film distinti e generi distinti + classe definita)
make_swrl_rule(
    "VersatileAuteurRule",
    "Regista di almeno un AcclaimedFilm che ha diretto 2 film distinti di 2 generi distinti → VersatileAuteur.",
    body_atoms=[
        cls_atom(CT.Regista, VAR_D),
        prop_atom(CT.directorOf, VAR_D, VAR_F1), cls_atom(CT.AcclaimedFilm, VAR_F1), prop_atom(CT.hasGenre, VAR_F1, VAR_G1),
        prop_atom(CT.directorOf, VAR_D, VAR_F2), prop_atom(CT.hasGenre, VAR_F2, VAR_G2),
        diff_atom(VAR_F1, VAR_F2), diff_atom(VAR_G1, VAR_G2),
    ],
    head_atoms=[cls_atom(CT.VersatileAuteur, VAR_D)],
)

# 5. SharedCollectionRule (senza self-loop grazie a DifferentIndividualsAtom)
make_swrl_rule(
    "SharedCollectionRule",
    "Due film distinti della stessa collezione condividono il franchise (evita self-loop a differenza di una property chain).",
    body_atoms=[
        cls_atom(CT.Film, VAR_F1), cls_atom(CT.Film, VAR_F2), diff_atom(VAR_F1, VAR_F2),
        prop_atom(CT.partOfCollection, VAR_F1, VAR_C), prop_atom(CT.partOfCollection, VAR_F2, VAR_C),
    ],
    head_atoms=[prop_atom(CT.sameCollectionAs, VAR_F1, VAR_F2)],
)

# 6. EnsembleAwardRule (chaining su 3 classi definite OWL 2 DL)
make_swrl_rule(
    "EnsembleAwardRule",
    "Film che è simultaneamente EnsembleFilm, AcclaimedFilm e ProfitableFilm → PremioFilm (chaining OWL 2 DL → SWRL).",
    body_atoms=[
        cls_atom(CT.Film, VAR_F), cls_atom(CT.EnsembleFilm, VAR_F),
        cls_atom(CT.AcclaimedFilm, VAR_F), cls_atom(CT.ProfitableFilm, VAR_F),
    ],
    head_atoms=[cls_atom(CT.PremioFilm, VAR_F)],
)

print("  → 6 regole SWRL aggiunte:")
print("    1. SelfDirectedRoleRule     2. FrequentCollaboratorRule   3. SequelDirectorRule")
print("    4. VersatileAuteurRule      5. SharedCollectionRule       6. EnsembleAwardRule")
print(f"  Triple totali dopo SWRL: {len(g)}")

[SWRL] Aggiunta di 6 regole DL-safe...
  → 6 regole SWRL aggiunte:
    1. SelfDirectedRoleRule     2. FrequentCollaboratorRule   3. SequelDirectorRule
    4. VersatileAuteurRule      5. SharedCollectionRule       6. EnsembleAwardRule
  Triple totali dopo SWRL: 14916


## Cella 7 — Serializzazione Multi-Formato (`core.ttl`, `core.owl`, `core.swrl.n3`)

Esportiamo l'ontologia in tre formati standard W3C (Lezioni 5–6 e 12):
- `data/ontology/core.ttl` (**Turtle**): leggibile e compatto, apribile direttamente in Protégé.
- `data/ontology/core.owl` (**RDF/XML**): formato nativo OWL API / Protégé (come `Frutta.owl` e `Researcher.rdf` della Lezione 12).
- `data/ontology/core.swrl.n3` (**N3**): le 7 regole SWRL isolate per ispezione e fallback.

In [12]:
# ── Cella 7: Serializzazione core.ttl, core.owl, core.swrl.n3 ─────────────
g.serialize(str(CORE_TTL), format="turtle")
g.serialize(str(CORE_OWL), format="xml")
swrl_g.serialize(str(CORE_SWRL_N3), format="n3")

print(f"✓ Scritto: {CORE_TTL} ({CORE_TTL.stat().st_size / 1024:.1f} KB, {len(g)} triple)")
print(f"✓ Scritto: {CORE_OWL} ({CORE_OWL.stat().st_size / 1024:.1f} KB, formato RDF/XML nativo Protégé)")
print(f"✓ Scritto: {CORE_SWRL_N3} ({len(swrl_g)} triple SWRL)")

✓ Scritto: C:\uni\CineTrope\data\ontology\core.ttl (477.3 KB, 14916 triple)
✓ Scritto: C:\uni\CineTrope\data\ontology\core.owl (1313.4 KB, formato RDF/XML nativo Protégé)
✓ Scritto: C:\uni\CineTrope\data\ontology\core.swrl.n3 (292 triple SWRL)


## Cella 8 — Reasoning OWL 2 DL + SWRL con HermiT via OWLReady2

Carichiamo `core.owl` in **OWLReady2** ed eseguiamo **HermiT** (`sync_reasoner_hermit(infer_property_values=True)`):
1. Verifica la **consistenza globale** dell'ontologia (`inconsistent_classes == 0`, nessuna classe equivalente a `owl:Nothing`).
2. Classifica tutti gli individui nelle **16 classi definite OWL 2 DL** (inclusa la prova OWA su `StandardLengthFilm`).
3. Esegue le **6 regole SWRL DL-safe** (congiunzione di ruoli, diamante a 4 nodi, triangolo sequel, chaining).
4. Propaga **gerarchie di proprietà**, **inverse**, **simmetrie** (incluso `coStarsWith` da `CONSTRUCT`), **chiusure transitive** (`sequelOf`, `subGenreOf`) e **property chain** (`sameDirectorAs`, `hasAncestorGenre`).

In [13]:
# ── Cella 8: HermiT reasoning ─────────────────────────────────────────────
def find_java() -> str | None:
    found = shutil.which("java")
    if found:
        return found
    search_roots = [
        ROOT / ".venv" / "jre",
        pathlib.Path.home() / ".jdk",
        pathlib.Path(r"C:\Program Files\Eclipse Adoptium"),
        pathlib.Path(r"C:\Program Files\Java"),
        pathlib.Path(r"C:\Program Files\Microsoft"),
    ]
    for base in search_roots:
        if base.exists():
            for exe in sorted(base.rglob("java.exe"), reverse=True):
                return str(exe)
    return None

java_exe = find_java()
onto = None
world = None
hermit_ok = False
inconsistent_classes = []

if java_exe:
    print(f"  Java trovato: {java_exe}")
    try:
        import owlready2
        owlready2.JAVA_EXE = java_exe

        world = owlready2.World()
        with open(CORE_OWL, "rb") as fh:
            onto = world.get_ontology("http://example.org/cinetrope").load(fileobj=fh)
        print("[HermiT] Avvio sync_reasoner_hermit(infer_property_values=True)...")
        t0 = time.time()
        with onto:
            owlready2.sync_reasoner_hermit(
                world,
                infer_property_values=True,
                debug=0,
            )
        elapsed = time.time() - t0
        inconsistent_classes = list(world.inconsistent_classes())
        print(f"✓ [HermiT] Reasoning completato in {elapsed:.1f}s")
        print(f"  Classi inconsistenti: {len(inconsistent_classes)}")
        if inconsistent_classes:
            print("  ⚠ Classi inconsistenti:", inconsistent_classes)
        hermit_ok = True
    except Exception as e:
        print(f"⚠ HermiT errore: {e}")
        onto = None
        hermit_ok = False
else:
    print("⚠ Java non trovato. HermiT non disponibile.")
    print("  Installa Java ≥ 11 oppure una JRE locale in .venv/jre.")

  Java trovato: C:\uni\CineTrope\.venv\jre\jdk-21.0.13.1+2-jre\bin\java.exe


[HermiT] Avvio sync_reasoner_hermit(infer_property_values=True)...


✓ [HermiT] Reasoning completato in 141.3s
  Classi inconsistenti: 0


## Cella 9 — Estrazione delle inferenze e materializzazione (`inferred.ttl` + `store.trig`)

Estraiamo tutte le asserzioni di classe e proprietà inferite da HermiT (filtrando i self-loop della property chain `sameDirectorAs`) e le salviamo in `data/generated/inferred.ttl`, poi rigeneriamo il dataset a named graph `data/generated/store.trig` per il backend FastAPI.

In [14]:
# ── Cella 9: Estrazione inferenze HermiT ──────────────────────────────────
DEFINED_AND_SWRL_CLASSES = [
    # Classi Film (OWL 2 DL + SWRL)
    CT.AcclaimedFilm, CT.Blockbuster, CT.EpicFilm, CT.StandardLengthFilm,
    CT.ProfitableFilm, CT.BoxOfficeFlop, CT.CollectionFilm, CT.MultiGenreFilm,
    CT.EnsembleFilm, CT.AwardedFilm, CT.AcclaimedBlockbuster, CT.NotableFilm,
    CT.PremioFilm,
    # Classi Persona (OWL 2 DL + SWRL)
    CT.Auteur, CT.VeteranDirector, CT.VeteranAuteur, CT.RegistaAttore,
    CT.AutoreInterprete, CT.SequelDirector, CT.VersatileAuteur,
]
INFERRED_OBJECT_PROPS = [
    CT.sameDirectorAs, CT.sameCollectionAs, CT.hasAncestorGenre,
    CT.directorOf, CT.actedIn, CT.hasPart,
    CT.hasContributor, CT.contributedTo,
    CT.sequelOf, CT.hasSequel,
    CT.hasFrequentCollaborator, CT.coStarsWith, CT.collaboratedWith,
    CT.relatedFilm,
]

inferred_g = Graph()
inferred_g.bind("ct", CT, override=True)
inferred_g.bind("rdf", RDF)
inferred_g.bind("rdfs", RDFS)
inferred_g.bind("owl", OWL)

class_counts = {}
prop_counts = {}

if hermit_ok and onto is not None and world is not None:
    # 1. Classi definite + SWRL
    for cls_iri in DEFINED_AND_SWRL_CLASSES:
        cls_name = str(cls_iri).split("#")[-1]
        cls_obj = world.search_one(iri=str(cls_iri))
        if cls_obj is None:
            class_counts[cls_name] = 0
            continue
        count = 0
        for ind in cls_obj.instances():
            ind_uri = URIRef(ind.iri)
            if not str(ind_uri).startswith(str(CT)):
                continue
            triple = (ind_uri, RDF.type, cls_iri)
            if triple not in g:
                inferred_g.add(triple)
            count += 1
        class_counts[cls_name] = count

    # 2. Object properties inferite (property chain, SWRL, inverse, transitive, subPropertyOf)
    for prop_iri in INFERRED_OBJECT_PROPS:
        prop_name = str(prop_iri).split("#")[-1]
        prop_obj = world.search_one(iri=str(prop_iri))
        count = 0
        if prop_obj is not None:
            for subj, obj in prop_obj.get_relations():
                s_uri, o_uri = URIRef(subj.iri), URIRef(obj.iri)
                if not (str(s_uri).startswith(str(CT)) and str(o_uri).startswith(str(CT))):
                    continue
                if prop_iri in (CT.sameDirectorAs, CT.sameCollectionAs, CT.relatedFilm, CT.coStarsWith, CT.collaboratedWith, CT.hasFrequentCollaborator) and s_uri == o_uri:
                    continue
                triple = (s_uri, prop_iri, o_uri)
                if triple not in g:
                    inferred_g.add(triple)
                    count += 1
        prop_counts[prop_name] = count

    # 3. Data property subPropertyOf inferita (pattern L10 NaviFinal: countryIsoCode / languageIsoCode -> isoCode)
    iso_count = 0
    for sub_p in (CT.countryIsoCode, CT.languageIsoCode):
        for s_uri, o_lit in g.subject_objects(sub_p):
            triple = (s_uri, CT.isoCode, o_lit)
            if triple not in g:
                inferred_g.add(triple)
                iso_count += 1
    prop_counts["isoCode (subProp)"] = iso_count

    print("[Inferenze] Classi definite OWL 2 DL & SWRL popolate da HermiT:")
    for cls_name, count in sorted(class_counts.items()):
        print(f"  {cls_name:26s}: {count}")
    print("\n[Inferenze] Proprietà inferite da HermiT:")
    for prop_name, count in sorted(prop_counts.items()):
        print(f"  {prop_name:26s}: {count}")
else:
    print("[Inferenze] HermiT non disponibile — file inferred.ttl vuoto (fallback owlrl attivo).")

inferred_g.serialize(str(INFERRED_TTL), format="turtle")
print(f"\n✓ Scritto: {INFERRED_TTL} ({len(inferred_g)} triple inferite)")

[Inferenze] Classi definite OWL 2 DL & SWRL popolate da HermiT:
  AcclaimedBlockbuster      : 16
  AcclaimedFilm             : 48
  Auteur                    : 64
  AutoreInterprete          : 7
  AwardedFilm               : 2
  Blockbuster               : 17
  BoxOfficeFlop             : 10
  CollectionFilm            : 24
  EnsembleFilm              : 97
  EpicFilm                  : 16
  MultiGenreFilm            : 61
  NotableFilm               : 49
  PremioFilm                : 29
  ProfitableFilm            : 41
  RegistaAttore             : 7
  SequelDirector            : 3
  StandardLengthFilm        : 83
  VersatileAuteur           : 6
  VeteranAuteur             : 18
  VeteranDirector           : 29

[Inferenze] Proprietà inferite da HermiT:
  actedIn                   : 970
  coStarsWith               : 54
  collaboratedWith          : 0
  contributedTo             : 0
  directorOf                : 127
  hasAncestorGenre          : 123
  hasContributor            : 0
  hasFr

In [15]:
# ── Cella 9b: Build store.trig + build_report.json per il backend FastAPI ─
from cinetrope.store import CineTropeStore

store = CineTropeStore.rebuild()
size_kb = STORE_PATH.stat().st_size / 1024
print(f"✓ Store aggiornato: {STORE_PATH} ({size_kb:.0f} KB)")
print("  Named graph:", [(str(s['source']).split('#')[-1], s['triples']) for s in store.source_stats()])

Store: carica core.ttl:   0%|          | 0/3 [00:00<?, ?fase/s]

Store: carica core.ttl:  33%|███▎      | 1/3 [00:01<00:03,  1.51s/fase]

Store: inferenze HermiT / fallback:  33%|███▎      | 1/3 [00:01<00:03,  1.51s/fase]

Store: inferenze HermiT / fallback:  67%|██████▋   | 2/3 [00:01<00:00,  1.42fase/s]

Store: validazione:  67%|██████▋   | 2/3 [00:01<00:00,  1.42fase/s]                

Store: validazione: 100%|██████████| 3/3 [00:01<00:00,  1.72fase/s]

✓ Store aggiornato: C:\uni\CineTrope\data\generated\store.trig (635 KB)
  Named graph: [('TMDB', 13555), ('Inferred', 2055), ('Ontology', 1225), ('DBpedia', 129), ('Wikidata', 7)]


## Cella 10 — Verifica SPARQL Completa (tutte le tecniche delle Lezioni 7–12)

Verifichiamo il Knowledge Graph esercitando **tutte le forme e clausole SPARQL** del corso:
1. **`SELECT` + aggregazione** sulle classi definite OWL 2 DL e SWRL;
2. **Prova empirica della Open World Assumption (OWA)** su `EpicFilm` vs `StandardLengthFilm` (`owl:complementOf` + `FunctionalProperty`);
3. **Property Paths (`+`)** su `subGenreOf+` e chiusura transitiva `sequelOf` (*Avengers*);
4. **Confronto OWL 2 DL vs SWRL**: `RegistaAttore` (`Regista ⊓ Attore`) vs `AutoreInterprete` (`SelfDirectedRoleRule`), più i diamanti `hasFrequentCollaborator` e `coStarsWith`;
5. **`ASK`** (consistenza `owl:Nothing` e inferenza specifica su Christopher Nolan ↔ Michael Caine);
6. **`CONSTRUCT`** di traduzione vocabolario `ct:` → `schema:`/`dbo:` (stile Lezione 8 `VCards`/`Directors`);
7. **`DESCRIBE`** su un individuo chiave (`ct:OscarBestPicture`);
8. **Query di esplorazione metadati/schema** (stile Lezione 7 slide 14–18).

In [16]:
# ── Cella 10: Verifica SPARQL completa (SELECT, ASK, CONSTRUCT, DESCRIBE) ─
print("[Verifica] Query SPARQL sullo store (core.ttl + inferred.ttl)...")
ds = store.dataset
PREFIX = (
    "PREFIX ct: <http://example.org/cinetrope#>\n"
    "PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>\n"
    "PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\n"
    "PREFIX owl: <http://www.w3.org/2002/07/owl#>\n"
    "PREFIX schema: <http://schema.org/>\n"
    "PREFIX dbo: <http://dbpedia.org/ontology/>\n"
)

# 1. SELECT con GROUP BY sulle classi definite e SWRL
q_classes = PREFIX + """
SELECT ?cls (COUNT(DISTINCT ?ind) AS ?n) WHERE {
  VALUES ?cls {
    ct:AcclaimedFilm ct:Blockbuster ct:EpicFilm ct:StandardLengthFilm
    ct:ProfitableFilm ct:BoxOfficeFlop ct:CollectionFilm ct:MultiGenreFilm
    ct:EnsembleFilm ct:AwardedFilm ct:AcclaimedBlockbuster ct:NotableFilm
    ct:PremioFilm
    ct:Auteur ct:VeteranDirector ct:VeteranAuteur ct:RegistaAttore
    ct:AutoreInterprete ct:SequelDirector ct:VersatileAuteur
  }
  ?ind rdf:type ?cls .
} GROUP BY ?cls ORDER BY DESC(?n)
"""
print("\n① [SELECT] Istanze classificate da HermiT (OWL 2 DL + SWRL):")
for row in ds.query(q_classes):
    print(f"  {str(row.cls).split('#')[-1]:24s}: {row.n}")

# 2. Dimostrazione OWA: EpicFilm vs StandardLengthFilm vs runtime ignoto
q_owa = PREFIX + """
SELECT ?title WHERE {
  ?f a ct:Film ; rdfs:label ?title .
  FILTER NOT EXISTS { ?f a ct:EpicFilm }
  FILTER NOT EXISTS { ?f a ct:StandardLengthFilm }
}
"""
owa_unknown = [str(r.title) for r in ds.query(q_owa)]
print(f"\n② [OWA Demo] Film né EpicFilm né StandardLengthFilm (runtime assente): {owa_unknown}")

# 3. Catena transitiva sequelOf + Property Path subGenreOf+
q_seq = PREFIX + """
SELECT ?f1Label ?f2Label WHERE {
  ?f2 ct:sequelOf ?f1 .
  ?f1 rdfs:label ?f1Label . ?f2 rdfs:label ?f2Label .
} ORDER BY ?f2Label ?f1Label
"""
print("\n③ [TransitiveProperty] Relazioni sequelOf inferite da HermiT:")
for r in ds.query(q_seq):
    print(f"  {r.f2Label}  --sequelOf-->  {r.f1Label}")

q_pp = PREFIX + """
SELECT ?subLabel ?ancLabel WHERE {
  ?sub ct:subGenreOf+ ?anc .
  ?sub rdfs:label ?subLabel . ?anc rdfs:label ?ancLabel .
} ORDER BY ?ancLabel ?subLabel LIMIT 8
"""
print("\n   [Property Path subGenreOf+] Campione gerarchia generi:")
for r in ds.query(q_pp):
    print(f"  {r.subLabel} ⊑+ {r.ancLabel}")

# 4. Diamanti SWRL: collaboratori frequenti & co-star ricorrenti
q_fc = PREFIX + """
SELECT ?dName ?aName WHERE {
  GRAPH ct:Inferred { ?d ct:hasFrequentCollaborator ?a . }
  ?d a ct:Regista ; rdfs:label ?dName .
  ?a a ct:Attore ; rdfs:label ?aName .
} ORDER BY ?dName ?aName LIMIT 8
"""
print("\n④ [SWRL Diamante] Campione hasFrequentCollaborator (regista ↔ attore ≥ 2 film):")
for r in ds.query(q_fc):
    print(f"  {r.dName} ↔ {r.aName}")

# 5. ASK booleane (L7)
ask_nothing = bool(ds.query(PREFIX + "ASK { ?x a owl:Nothing }"))
ask_nolan_caine = bool(ds.query(PREFIX + "ASK { ct:ChristopherNolan ct:hasFrequentCollaborator ct:MichaelCaine }"))
ask_avengers_trans = bool(ds.query(PREFIX + "ASK { ct:TMDB1003596 ct:sequelOf ct:TMDB299536 }"))
print(f"\n⑤ [ASK] Inconsistenze (owl:Nothing)?                    {ask_nothing}")
print(f"   [ASK] ChristopherNolan hasFrequentCollaborator Caine? {ask_nolan_caine}")
print(f"   [ASK] Avengers Doomsday sequelOf Infinity War?        {ask_avengers_trans}")

# 6. CONSTRUCT di traduzione vocabolario ct: -> schema: / dbo: (stile L8 VCards)
q_construct = PREFIX + """
CONSTRUCT {
  ?f a schema:Movie, dbo:Film ;
     schema:name ?t ;
     schema:director ?d .
  ?d a dbo:MovieDirector ; rdfs:label ?dn .
}
WHERE {
  BIND(ct:TMDB27205 AS ?f)
  ?f ct:title ?t ; ct:directedBy ?d .
  ?d rdfs:label ?dn .
}
"""
mapped_g = ds.query(q_construct).graph
print(f"\n⑥ [CONSTRUCT L8] Proiezione ct:TMDB27205 verso schema.org + DBpedia ({len(mapped_g)} triple):")
print(mapped_g.serialize(format="turtle").strip())

# 7. DESCRIBE (L7)
desc_g = ds.query(PREFIX + "DESCRIBE ct:OscarBestPicture").graph
print(f"\n⑦ [DESCRIBE L7] Descrizione di ct:OscarBestPicture ({len(desc_g)} triple):")
for s, p, o in list(desc_g)[:6]:
    print(f"  {str(s).split('#')[-1]}  {str(p).split('#')[-1].split('/')[-1]}  {o}")

# 8. Query metadati/schema (L7 slide 14-18)
q_meta = PREFIX + """
SELECT (COUNT(DISTINCT ?c) AS ?classes) (COUNT(DISTINCT ?op) AS ?objProps) (COUNT(DISTINCT ?dp) AS ?dataProps) WHERE {
  { ?c a owl:Class . FILTER(STRSTARTS(STR(?c), STR(ct:))) }
  UNION { ?op a owl:ObjectProperty . FILTER(STRSTARTS(STR(?op), STR(ct:))) }
  UNION { ?dp a owl:DatatypeProperty . FILTER(STRSTARTS(STR(?dp), STR(ct:))) }
}
"""
meta_row = list(ds.query(q_meta))[0]
print(f"\n⑧ [Metadata Query L7] Termini TBox ct:: {meta_row.classes} classi, {meta_row.objProps} ObjectProperty, {meta_row.dataProps} DatatypeProperty")

if ask_nothing or inconsistent_classes:
    print("\n⚠ INCONSISTENZA RILEVATA!")
else:
    print("\n✓ Ontologia 100% consistente (0 classi o individui owl:Nothing)")

[Verifica] Query SPARQL sullo store (core.ttl + inferred.ttl)...

① [SELECT] Istanze classificate da HermiT (OWL 2 DL + SWRL):
  EnsembleFilm            : 97
  StandardLengthFilm      : 83
  Auteur                  : 64
  MultiGenreFilm          : 61
  NotableFilm             : 49
  AcclaimedFilm           : 48
  ProfitableFilm          : 41
  PremioFilm              : 29
  VeteranDirector         : 29
  CollectionFilm          : 24
  VeteranAuteur           : 18
  Blockbuster             : 17
  EpicFilm                : 16
  AcclaimedBlockbuster    : 16
  BoxOfficeFlop           : 10
  RegistaAttore           : 7
  AutoreInterprete        : 7
  VersatileAuteur         : 6
  SequelDirector          : 3
  AwardedFilm             : 2

② [OWA Demo] Film né EpicFilm né StandardLengthFilm (runtime assente): ['Insecta']

③ [TransitiveProperty] Relazioni sequelOf inferite da HermiT:
  Avengers: Doomsday  --sequelOf-->  Avengers: Endgame
  Avengers: Doomsday  --sequelOf-->  Avengers: Infinity 


⑧ [Metadata Query L7] Termini TBox ct:: 35 classi, 31 ObjectProperty, 24 DatatypeProperty

✓ Ontologia 100% consistente (0 classi o individui owl:Nothing)


## Cella 11 — Riepilogo Finale per Ispezione in Protégé

In [17]:
# ── Cella 11: Riepilogo finale ────────────────────────────────────────────
print("=" * 68)
print("RIEPILOGO ONTOLOGIA CineTrope v3.1 (OWL 2 DL / SROIQ(D) + SWRL)")
print("=" * 68)
print(f"  Turtle (Protégé):  {CORE_TTL} ({CORE_TTL.stat().st_size / 1024:.1f} KB)")
print(f"  RDF/XML (Protégé): {CORE_OWL} ({CORE_OWL.stat().st_size / 1024:.1f} KB)")
print(f"  SWRL (N3):         {CORE_SWRL_N3}")
print(f"  Inferenze HermiT:  {INFERRED_TTL} ({len(inferred_g)} triple)")
print(f"  Store TriG:        {STORE_PATH} ({STORE_PATH.stat().st_size / 1024:.0f} KB)")
print(f"  Triple core.ttl:   {len(g)}")
print(f"  Reasoner:          HermiT via OWLReady2 ({'OK, 0 inconsistenze' if hermit_ok and not inconsistent_classes else 'FALLBACK'})")
print()
print("Costruttori OWL 2 DL (SROIQ(D)) esercitati:")
print("  • owl:disjointUnionOf + owl:AllDisjointClasses + owl:disjointWith")
print("  • owl:allValuesFrom (condizioni necessarie su Film)")
print("  • owl:someValuesFrom (su classe nominata, definita e datatype)")
print("  • rdfs:Datatype restrictions (xsd:minInclusive, maxInclusive, minExclusive, maxExclusive)")
print("  • owl:minQualifiedCardinality + owl:AllDifferent (MultiGenreFilm)")
print("  • owl:hasValue (AwardedFilm — Wikidata)")
print("  • owl:intersectionOf (AcclaimedBlockbuster, RegistaAttore, VeteranAuteur)")
print("  • owl:unionOf (NotableFilm)")
print("  • owl:complementOf + owl:FunctionalProperty (StandardLengthFilm — prova OWA)")
print("  • owl:propertyChainAxiom (sameDirectorAs, hasAncestorGenre)")
print("  • owl:TransitiveProperty, AsymmetricProperty, IrreflexiveProperty, SymmetricProperty, inverseOf")
print("  • owl:sameAs (Linked Open Data verso DBpedia dbr:, Wikidata wd:, alias SciFi)")
print("  • 6 regole SWRL DL-safe (congiunzione di ruoli, diamante a 4 nodi, triangolo sequel, chaining)")
print("  • 1 regola SPARQL CONSTRUCT L8 (coStarsWith unidirezionale → simmetria inferita da HermiT)")

RIEPILOGO ONTOLOGIA CineTrope v3.1 (OWL 2 DL / SROIQ(D) + SWRL)
  Turtle (Protégé):  C:\uni\CineTrope\data\ontology\core.ttl (477.3 KB)
  RDF/XML (Protégé): C:\uni\CineTrope\data\ontology\core.owl (1313.4 KB)
  SWRL (N3):         C:\uni\CineTrope\data\ontology\core.swrl.n3
  Inferenze HermiT:  C:\uni\CineTrope\data\generated\inferred.ttl (2055 triple)
  Store TriG:        C:\uni\CineTrope\data\generated\store.trig (635 KB)
  Triple core.ttl:   14916
  Reasoner:          HermiT via OWLReady2 (OK, 0 inconsistenze)

Costruttori OWL 2 DL (SROIQ(D)) esercitati:
  • owl:disjointUnionOf + owl:AllDisjointClasses + owl:disjointWith
  • owl:allValuesFrom (condizioni necessarie su Film)
  • owl:someValuesFrom (su classe nominata, definita e datatype)
  • rdfs:Datatype restrictions (xsd:minInclusive, maxInclusive, minExclusive, maxExclusive)
  • owl:minQualifiedCardinality + owl:AllDifferent (MultiGenreFilm)
  • owl:hasValue (AwardedFilm — Wikidata)
  • owl:intersectionOf (AcclaimedBlockbuster, Re